# 🇯🇵 Japanese Customer-Support ASR with EmbeddingGemma 2 (CTC reader)

**Fine-tune Google's multimodal embedding model `google/embeddinggemma-2` into a fast, non-autoregressive
Japanese speech-to-text system for contact-center audio.**

This notebook adapts the approach of Quan Nguyen's [`qnguyen3/gemma-asr`](https://github.com/qnguyen3/gemma-asr)
(Vietnamese, state of the art with ~3,000 h in ~2 h on 8×H100): keep the EmbeddingGemma 2 **audio encoder frozen**
and train only a small **CTC "reader"** (~22 M params) on top of it.

```
16 kHz wav ─► log-mel (GPU, identical to the Gemma feature extractor)
          ─► EmbeddingGemma 2 audio encoder (USM-style Conformer, ~300M, 12 layers, 25 frames/s)   [frozen]
          ─► learned softmax mix of its 13 layer outputs
          ─► reader: Linear + depthwise conv + 6 × transformer (d=512)                            [~22M, trained]
          ─► head over the Gemma tokens that occur in Japanese text (rows initialised from Gemma's own
             token embeddings) ─► CTC ─► greedy decoding
```

### What this notebook gives you

| Part | What you get |
|---|---|
| 1–2 | Setup, config, and a **DRY_RUN** mode that runs the whole pipeline on CPU with a stand-in encoder |
| 3 | **Japanese text normalization** (NFKC, punctuation, spacing, numbers → kanji, fillers) with tests |
| 4 | **Customer-support domain kit**: glossary, keigo phrases, intents, escalation terms, PII patterns |
| 5–6 | **Data preparation** (Common Voice ja, FLEURS ja, JSUT, ReazonSpeech, *your own call recordings*) into repo-compatible parquet shards + data audit |
| 7 | **CTC unit vocabulary** from Gemma tokens, with glossary coverage check |
| 8–9 | **Model** (frozen encoder + reader) and **telephony augmentation** (8 kHz narrowband, μ-law, noise) |
| 10–12 | **Training** with CER-based model selection, curves, layer-weight inspection |
| 13 | **Evaluation**: CER, kana-CER, sentence error rate, confusions, product-name / number recall, telephone condition |
| 14–15 | **Inference** on long audio + **post-processing** (numbers → digits, glossary correction, PII masking) |
| 16 | **Stereo call transcription** (operator / customer channels) → timestamped dialogue |
| 17 | **Call analytics**: intents, escalation risk, keigo compliance, talk ratio |
| 18 | **Domain adaptation** (stage-2 training on in-domain calls, light encoder unfreezing) |
| 19 | Running the **original repo scripts** (multi-GPU `torchrun`) on the data prepared here |
| 20 | **Serving**: LitServe API + MCP tool (fits the rest of this repository) |
| 21 | Troubleshooting, privacy (APPI), and references |

> **Why CER?** Japanese is written without spaces and mixes kanji / hiragana / katakana, so "words" are not well
> defined. Character Error Rate on space-stripped text is the standard metric (as used for CSJ, JSUT,
> ReazonSpeech, and Japanese Whisper evaluations).

### Hardware

| Setup | What is realistic |
|---|---|
| CPU only | `DRY_RUN=True` smoke test of every cell (stand-in encoder, synthetic audio) |
| 1× RTX 4090 / A6000 (24–48 GB) | Common Voice + FLEURS baseline (~tens of hours) in well under an hour; domain fine-tuning |
| 1× A100 / H100 | ReazonSpeech subsets (hundreds of hours) |
| 4–8× H100 | Thousands of hours via `torchrun` (Part 19) |

Memory is modest because the 300M encoder runs frozen in bf16 (the reference run used ~9.5 GB at 600 s of audio per batch).

The code in this notebook is written to `ja_asr/*.py` files with `%%writefile`, so after one run you have a
reusable Python package for training, batch transcription and serving.

---
## 1. Installation

Recommended: Python 3.11+, a CUDA GPU and `transformers>=5.19` (EmbeddingGemma 2 support).
`google/embeddinggemma-2` is a gated model: accept the license on its Hugging Face page and log in with `hf auth login`
(or set `HF_TOKEN`).

Uncomment the install cell on a fresh machine. Match the torch wheel to your CUDA driver.

In [ ]:
# !pip install torch --index-url https://download.pytorch.org/whl/cu126
# !pip install "transformers>=5.19" "huggingface_hub[hf_xet]" datasets pyarrow "soundfile>=0.12" soxr numpy \
#              "jiwer>=3.0" sentencepiece tokenizers pandas matplotlib
# Optional extras:
# !pip install pykakasi          # kana-level CER (reading-based, forgives kanji/kana spelling variants)
# !pip install litserve fastmcp httpx   # serving (Part 20)
# !hf auth login

In [ ]:
import os, sys, json, math, time, io, re, random, glob, platform, importlib
from pathlib import Path

import numpy as np
import torch

print("python      ", platform.python_version())
print("torch       ", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| GPUs:", torch.cuda.device_count())
for name in ["transformers", "datasets", "soundfile", "soxr", "pyarrow", "jiwer", "pykakasi", "litserve"]:
    try:
        mod = importlib.import_module(name)
        print(f"{name:12s}", getattr(mod, "__version__", "ok"))
    except ImportError:
        print(f"{name:12s} (not installed)")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.0f} GB")

---
## 2. Configuration

Everything you would normally change lives in `CFG`.

**`DRY_RUN`** (default: on when no GPU is present) replaces EmbeddingGemma 2 with a tiny random "encoder", the
Gemma tokenizer with a character tokenizer, and the public datasets with synthetic audio, so that **every cell
runs end to end on a laptop**. The numbers it prints only prove that the plumbing works. Set
`JA_ASR_DRY_RUN=0` (or edit the cell) for real training.

In [ ]:
HAS_CUDA = torch.cuda.is_available()
_env = os.environ.get("JA_ASR_DRY_RUN", "auto").lower()
DRY_RUN = (not HAS_CUDA) if _env == "auto" else _env in ("1", "true", "yes")

CFG = dict(
    encoder_id="dummy" if DRY_RUN else "google/embeddinggemma-2",
    work_dir="ja_asr_work",
    prepared_dir="ja_asr_work/prepared",       # parquet shards: <prepared_dir>/<split>/<source>/*.parquet
    raw_dir="ja_asr_work/raw",                 # raw HF downloads
    run_dir="ja_asr_work/runs/ja_general",     # stage-1 checkpoints
    domain_run_dir="ja_asr_work/runs/ja_support",  # stage-2 (domain) checkpoints

    # ---- data
    public_sources=["cv17", "fleurs"],         # add "jsut", "reazonspeech_small" ... (see Part 5)
    max_files_per_source=0,                    # 0 = all parquet files; small number for a quick test
    local_corpus=None,                         # e.g. "data/support_calls/manifest.csv" (Part 5.3)
    min_dur=0.5, max_dur=30.0,
    digit_policy="kanji",                      # "drop" | "kanji" | "keep"  (Part 3)
    remove_fillers=False,                      # strip えー / えーと / あのー from labels (and hypotheses)

    # ---- vocabulary
    min_count=2,                               # rarer Gemma tokens -> <unk>

    # ---- model (reader)
    d_model=512, n_layers=6, n_heads=8, dropout=0.1,
    unfreeze_top_k=0,                          # stage 1: fully frozen encoder

    # ---- training
    max_steps=10_000, warmup_steps=1000,
    max_batch_seconds=600.0,                   # padded audio per step (per GPU)
    lr=1e-3, encoder_lr=5e-5, weight_decay=0.01, grad_clip=1.0, min_lr_ratio=0.05,
    telephony_aug_prob=0.5,                    # fraction of training utterances degraded to phone quality
    num_workers=4,
    log_every=50, eval_every=1000, save_every=1000, max_eval_utts=1000,
    seed=1234,
)

if DRY_RUN:  # tiny everything so the notebook runs on CPU in a few minutes
    CFG.update(public_sources=[], d_model=128, n_layers=2, n_heads=4, max_steps=400, warmup_steps=40,
               max_batch_seconds=60.0, num_workers=0, log_every=25, eval_every=100, save_every=100,
               telephony_aug_prob=0.3, min_count=1)

for k in ("work_dir", "prepared_dir", "raw_dir", "run_dir", "domain_run_dir"):
    Path(CFG[k]).mkdir(parents=True, exist_ok=True)
Path("ja_asr").mkdir(exist_ok=True)
Path("ja_asr/__init__.py").write_text('"""Japanese customer-support ASR on EmbeddingGemma 2 (CTC reader)."""\n')
if "." not in sys.path:
    sys.path.insert(0, ".")

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
print("DRY_RUN =", DRY_RUN, "| device =", DEVICE)
print(json.dumps(CFG, indent=1, ensure_ascii=False))

In [ ]:
# Re-import ja_asr modules automatically after they are rewritten by %%writefile
%load_ext autoreload
%autoreload 2

---
## 3. Japanese text normalization

CTC learns a deterministic mapping from sound to symbols, so **every spelling variant that sounds the same costs
accuracy**. The same normalization is applied to training labels, evaluation references, and model output.

| Step | Why | Example |
|---|---|---|
| Unicode **NFKC** | full-width Latin/digits → ASCII, half-width katakana → full-width | `ＡＢＣ１２３ ｶﾀｶﾅ` → `abc123 カタカナ` |
| lowercase | mixed English terms (`iPhone`, `Wi-Fi`) | `wifi` |
| remove punctuation & symbols | not audible; CTC has no punctuation model | `。、！？「」（）・〜♪` |
| **keep `ー`** | the long-vowel mark *is* audible (`サーバー`) | — |
| spacing | Japanese has no spaces; keep a space only between two Latin/digit tokens | `お客様 の id` → `お客様のid` |
| **numbers** | digits do not say how they were spoken | `3,000円` → `三千円` (policy `kanji`) or drop the utterance (`drop`) |
| fillers (optional) | `えー`, `えーと`, `あのー` are inconsistently transcribed across corpora | — |
| reject odd scripts | Hangul, emoji, etc. usually mean a bad label | → `None` |

**Numbers in customer support** matter (prices, order numbers, phone numbers). Common corpora write them as digits,
kanji or a mix. The `kanji` policy converts digits to their most likely reading (`1500` → `千五百`, digit-by-digit for
long IDs like `0312345678` → `〇三一二三四五六七八`). At inference time Part 15 converts kanji numerals back to
digits (inverse text normalization). For your own data, write numbers **as spoken** in the transcription guideline.

**Keigo** (polite endings such as `〜でございます`, `〜いたします`) is kept verbatim: it is real speech, and stripping it
would make labels inconsistent.

In [ ]:
%%writefile ja_asr/text.py
"""Japanese transcript normalization for CTC training and CER scoring.

Every transcript (train and eval) and every model hypothesis goes through the same functions so the model never
has to learn spelling variants that sound identical.
"""

import re
import unicodedata
from functools import lru_cache

KANJI_DIGITS = "〇一二三四五六七八九"
_DIGIT_ANY = re.compile(r"\d")
_THOUSANDS_SEP = re.compile(r"(?<=\d),(?=\d{3}(?!\d))")
_NUMBER = re.compile(r"\d+(?:\.\d+)?")
_DIGIT_GROUPS = re.compile(r"\d+(?:-\d+)+")  # 03-1234-5678 style: read digit by digit
# Hesitation fillers (only unambiguous forms; "あの" / "その" / "まあ" are also real words and are kept)
_FILLER = re.compile(r"(?:えーっと|えーと|えっと|ええと|えー+|あのー+|そのー+|うーん|んー+)")


@lru_cache(maxsize=65536)
def is_japanese_char(c: str) -> bool:
    o = ord(c)
    return (
        0x3040 <= o <= 0x309F      # hiragana
        or 0x30A0 <= o <= 0x30FF   # katakana (incl. ー)
        or 0x31F0 <= o <= 0x31FF   # katakana phonetic extensions
        or 0x3400 <= o <= 0x4DBF   # CJK ext A
        or 0x4E00 <= o <= 0x9FFF   # CJK unified ideographs
        or 0xF900 <= o <= 0xFAFF   # CJK compatibility ideographs
        or c in "々〆〇ヶヵ"
    )


def char_type(c: str) -> str:
    o = ord(c)
    if 0x3040 <= o <= 0x309F:
        return "hiragana"
    if 0x30A0 <= o <= 0x30FF or 0x31F0 <= o <= 0x31FF:
        return "katakana"
    if is_japanese_char(c):
        return "kanji"
    if c.isascii() and c.isalpha():
        return "latin"
    if c.isdigit():
        return "digit"
    return "other"


# ------------------------------------------------------------------ numbers
def _under_10k(n: int) -> str:
    out = ""
    for val, ch in ((1000, "千"), (100, "百"), (10, "十")):
        q, n = divmod(n, val)
        if q:
            out += ("" if q == 1 else KANJI_DIGITS[q]) + ch
    return out + (KANJI_DIGITS[n] if n else "")


def int_to_kanji(n: int) -> str:
    """1500 -> 千五百, 20000 -> 二万, 0 -> 〇 (positional reading)."""
    if n == 0:
        return "〇"
    out = ""
    for val, ch in ((10**16, "京"), (10**12, "兆"), (10**8, "億"), (10**4, "万")):
        q, n = divmod(n, val)
        if q:
            out += _under_10k(q) + ch
    return out + _under_10k(n)


def number_to_kanji(s: str, digitwise_min_len: int = 5) -> str:
    """Most likely reading of a digit string as it appears in transcripts.

    * decimals: 3.5 -> 三点五
    * IDs / phone numbers (leading zero or >= digitwise_min_len digits that are not round): read digit by digit
    * otherwise positional: 1500 -> 千五百
    """
    if "." in s:
        a, b = s.split(".", 1)
        return number_to_kanji(a, digitwise_min_len) + "点" + "".join(KANJI_DIGITS[int(d)] for d in b)
    round_number = len(s) > 1 and s.rstrip("0") != s and len(s.rstrip("0")) <= 2
    if (len(s) > 1 and s[0] == "0") or (len(s) >= digitwise_min_len and not round_number):
        return "".join(KANJI_DIGITS[int(d)] for d in s)
    return int_to_kanji(int(s))


# ------------------------------------------------------------------ spacing / characters
def _strip_symbols(s: str) -> str:
    """Replace punctuation, symbols, separators and control chars by spaces; keep letters, numbers, marks, ー."""
    return "".join(c if (c == "ー" or unicodedata.category(c)[0] in "LNM") else " " for c in s)


def join_tokens(tokens: list[str]) -> str:
    """Japanese has no spaces: keep a space only between two ASCII (Latin/digit) tokens."""
    if not tokens:
        return ""
    out = tokens[0]
    for t in tokens[1:]:
        out += (" " + t) if (out[-1].isascii() and t[0].isascii()) else t
    return out


def tidy_spaces(text: str) -> str:
    return join_tokens(text.split())


def _valid_char(c: str) -> bool:
    return is_japanese_char(c) or (c.isascii() and c.isalnum())


# ------------------------------------------------------------------ public API
def normalize_japanese(text: str, digit_policy: str = "kanji", remove_fillers: bool = False) -> str | None:
    """Normalize a training/eval transcript. Returns None if the utterance should be discarded.

    digit_policy: "drop" (discard utterances containing digits), "kanji" (convert to likely reading), "keep".
    """
    if not text:
        return None
    s = unicodedata.normalize("NFKC", text).lower()
    s = re.sub(r"(?<=[ぁ-んァ-ン])[〜~]", "ー", s)  # 「すご〜い」 style lengthening is a long vowel
    if _DIGIT_ANY.search(s):
        if digit_policy == "drop":
            return None
        if digit_policy == "kanji":
            s = _THOUSANDS_SEP.sub("", s)
            s = _DIGIT_GROUPS.sub(lambda m: "".join(KANJI_DIGITS[int(d)] for d in m.group() if d != "-"), s)
            s = _NUMBER.sub(lambda m: number_to_kanji(m.group()), s)
    s = _strip_symbols(s)
    if remove_fillers:
        s = _FILLER.sub(" ", s)
    tokens = s.split()
    if not tokens:
        return None
    if not all(_valid_char(c) for t in tokens for c in t):
        return None
    return join_tokens(tokens)


def normalize_hypothesis(text: str, remove_fillers: bool = False) -> str:
    """Lenient normalization for model output / scoring (never rejects, drops odd characters)."""
    s = _strip_symbols(unicodedata.normalize("NFKC", text or "").lower())
    if remove_fillers:
        s = _FILLER.sub(" ", s)
    s = "".join(c if (_valid_char(c) or c == " ") else " " for c in s)
    return join_tokens(s.split())


def cer_text(text: str) -> str:
    """The string CER is computed on: spaces removed."""
    return text.replace(" ", "")


def hira_to_kata(s: str) -> str:
    return "".join(chr(ord(c) + 0x60) if 0x3041 <= ord(c) <= 0x3096 else c for c in s)


def kata_to_hira(s: str) -> str:
    return "".join(chr(ord(c) - 0x60) if 0x30A1 <= ord(c) <= 0x30F6 else c for c in s)


# Backwards-compatible names used by qnguyen3/gemma-asr (see Part 19)
def normalize_text(text: str, drop_digits: bool = False) -> str | None:
    return normalize_japanese(text, digit_policy="drop" if drop_digits else "kanji")


def strip_tones(text: str) -> str:  # Japanese has no tone marks; identity keeps the repo's scoring code working
    return text

In [ ]:
from ja_asr.text import (normalize_japanese, normalize_hypothesis, number_to_kanji, int_to_kanji,
                         cer_text, char_type, hira_to_kata, kata_to_hira)

tests = [
    ("お電話ありがとうございます。サクラモバイル、カスタマーセンターの田中でございます。", "kanji",
     "お電話ありがとうございますサクラモバイルカスタマーセンターの田中でございます"),
    ("ＩＤとパスワードを入力してください！", "kanji", "idとパスワードを入力してください"),
    ("ｶﾀｶﾅのﾃｽﾄです", "kanji", "カタカナのテストです"),
    ("月額3,000円のプランです", "kanji", "月額三千円のプランです"),
    ("お電話番号は03-1234-5678ですね", "kanji", "お電話番号は〇三一二三四五六七八ですね"),
    ("ご注文番号は12345でございます", "kanji", "ご注文番号は一二三四五でございます"),
    ("料金は1500円、2.5ギガです", "kanji", "料金は千五百円二点五ギガです"),
    ("月額3,000円のプランです", "drop", None),
    ("Wi-Fi が つながらない", "kanji", "wi fiがつながらない"),
    ("「解約」をご希望ですか？", "kanji", "解約をご希望ですか"),
    ("すご〜い、サーバーが速い", "kanji", "すごーいサーバーが速い"),
    ("안녕하세요", "kanji", None),
]
ok = True
for src, pol, want in tests:
    got = normalize_japanese(src, digit_policy=pol)
    flag = "✓" if got == want else "✗"
    ok &= got == want
    print(f"{flag} [{pol:5s}] {src!r:48} -> {got!r}")
print("\nfiller removal:", normalize_japanese("えーと、あのー、解約したいんですけど", remove_fillers=True))
print("int_to_kanji  :", [int_to_kanji(n) for n in (10, 15, 100, 1000, 3000, 10000, 123456789)])
print("hira<->kata   :", hira_to_kata("ぎがぷらん"), kata_to_hira("ギガプラン"))
assert ok, "normalization tests failed"

---
## 4. Customer-support domain kit

A contact-center ASR is judged on the words that matter to the business: **product and plan names, company
names, numbers, and the intent of the call**. Collect these in one place; they are used for

* **vocabulary coverage** (Part 7): make sure product names are not split into `<unk>`,
* **evaluation** (Part 13): term recall on product names and numbers, not just overall CER,
* **post-correction** (Part 15): map known misrecognitions to the canonical spelling,
* **analytics** (Part 17): intents, escalation risk, keigo compliance.

The company and products below (サクラモバイル, ギガプラン …) are **fictional examples**; replace them with your own.

In [ ]:
GLOSSARY = {
    # canonical spelling : known variants / misrecognitions (add the ones you see in Part 13)
    "サクラモバイル": ["さくらもばいる", "桜モバイル"],
    "ギガプラン": ["ぎがぷらん", "ギガプラント"],
    "かけ放題オプション": ["掛け放題オプション", "かけほうだいオプション"],
    "サクラ光": ["さくらひかり", "桜光"],
    "マイページ": ["まいぺーじ", "my page"],
    "esim": ["イーシム", "いーしむ"],
    "simカード": ["シムカード", "しむかーど"],
    "機種変更": ["きしゅへんこう"],
    "名義変更": ["めいぎへんこう"],
    "契約者": [],
    "暗証番号": [],
    "口座振替": ["口座振り替え"],
}

KEIGO_PHRASES = [
    "お電話ありがとうございます", "少々お待ちください", "恐れ入りますが", "申し訳ございません",
    "かしこまりました", "確認いたします", "お待たせいたしました", "担当の者に代わります",
    "ご本人様確認", "よろしいでしょうか", "失礼いたします", "ご不便をおかけして",
]

# Sample support utterances: used for vocabulary checks, the DRY_RUN corpus, and analytics demos.
SUPPORT_SENTENCES = [
    ("agent", "お電話ありがとうございますサクラモバイルカスタマーセンターの田中でございます"),
    ("customer", "すみません先月の請求額がいつもより高いんですけど"),
    ("agent", "ご不便をおかけして申し訳ございませんご契約者様のお名前を伺ってもよろしいでしょうか"),
    ("customer", "山田太郎です"),
    ("agent", "ありがとうございますご本人様確認のため生年月日をお願いいたします"),
    ("customer", "昭和六十年五月三日です"),
    ("agent", "確認いたしますので少々お待ちください"),
    ("agent", "お待たせいたしました先月はかけ放題オプションの料金が追加されております"),
    ("customer", "そんなオプション申し込んだ覚えがないんですが"),
    ("agent", "大変失礼いたしましたお申し込み履歴を確認いたします"),
    ("customer", "解約したいんですけどどうすればいいですか"),
    ("agent", "解約のお手続きはマイページまたはお電話で承っております"),
    ("customer", "simカードが届かないんですが"),
    ("agent", "配送状況をお調べいたしますご注文番号をお願いいたします"),
    ("customer", "インターネットがつながらないんです"),
    ("agent", "ルーターの電源を一度抜いていただけますでしょうか"),
    ("customer", "機種変更したらesimが使えなくなりました"),
    ("agent", "かしこまりましたesimの再発行のお手続きをご案内いたします"),
    ("customer", "ギガプランに変更したいです"),
    ("agent", "ギガプランは月額三千円でご利用いただけます"),
    ("customer", "口座振替に変えたいんですけど"),
    ("agent", "お支払い方法の変更ですね書類をお送りいたします"),
    ("customer", "責任者を出してください"),
    ("agent", "申し訳ございません担当の者に代わります"),
    ("customer", "サクラ光の工事日を変更したいです"),
    ("agent", "工事日の変更を承りました"),
    ("customer", "暗証番号を忘れてしまいました"),
    ("agent", "暗証番号の再設定をご案内いたします"),
    ("customer", "名義変更をお願いしたいです"),
    ("agent", "名義変更には本人確認書類が必要でございます"),
    ("customer", "返品したいんですけど"),
    ("agent", "返品のお手続きを承りますお届けから八日以内でございます"),
]
print(len(GLOSSARY), "glossary terms |", len(KEIGO_PHRASES), "keigo phrases |", len(SUPPORT_SENTENCES), "sample utterances")

---
## 5. Data preparation

All data ends up in the format used by `qnguyen3/gemma-asr`, so the original training scripts (Part 19) can
read it too:

```
<prepared_dir>/<split>/<source>/<shard>.parquet
    audio: FLAC bytes (16 kHz mono, PCM_16)   text: normalized transcript   duration: float32 s   source: str
```

### 5.1 Public Japanese corpora

| key | dataset | hours (approx.) | notes |
|---|---|---|---|
| `cv17` | Common Voice 17 ja (parquet mirror `fixie-ai/common_voice_17_0`) | ~70 validated | multi-speaker read speech, CC0 |
| `fleurs` | `google/fleurs` `ja_jp` | ~10 | read Wikipedia sentences, clean |
| `jsut` | JSUT basic5000 (`japanese-asr/ja_asr.jsut_basic5000`) | ~7 | single speaker; good clean **test** set |
| `reazonspeech_*` | `reazon-research/reazonspeech` (tiny/small/medium/large/all) | 8.5 h … 35,000 h | TV broadcast speech; the big one for strong results |

Hugging Face repos move and change layout; if a download fails, check the dataset page and adjust the
`repo` / glob patterns below. Gated datasets need you to accept their terms with the account used by `hf auth login`.

### 5.2 Customer-support data

Public data gives you general Japanese. Accuracy on *your* calls comes from in-domain audio:

* **Your own (anonymized, consented) call recordings** — even 20–50 hours help a lot with product names, keigo and
  telephone acoustics. See the transcription guideline at the end of this part.
* **CALLS** (empathetic customer-center dialogue corpus, research use) and other call-center style dialogue sets.
* **Synthetic** calls: TTS of real (anonymized) chat logs in several voices, degraded to 8 kHz.

### 5.3 Local corpus format

Either a folder with `xxx.wav|flac|mp3` + `xxx.txt` pairs, or a CSV/TSV manifest:

```
path,text,call_id,speaker,split
calls/0001_agent_003.wav,お電話ありがとうございます…,0001,agent,
```

`split` is optional: when empty, utterances are assigned to train/valid/test **by `call_id`** (hash), so the same
call never appears in both train and test (which would inflate scores).

In [ ]:
%%writefile ja_asr/data.py
"""Data preparation, telephony augmentation and loaders (shard format compatible with qnguyen3/gemma-asr)."""

import csv
import fnmatch
import glob
import io
import json
import os
import random
import zlib
from collections import Counter

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf
import soxr
import torch
from torch.utils.data import IterableDataset, get_worker_info

SR = 16000
FRAME_RATE = 25  # encoder output frames per second (40 ms)
ROW_GROUP = 512
SCHEMA = pa.schema([("audio", pa.binary()), ("text", pa.string()), ("duration", pa.float32()),
                    ("source", pa.string())])

# split -> glob patterns inside the HF dataset repo ("parquet" mode, like the original prepare_data.py)
PUBLIC_SOURCES = {
    "cv17": dict(repo="fixie-ai/common_voice_17_0", text="sentence",
                 splits={"train": ["ja/train/*/*.parquet"], "valid": ["ja/validation/*/*.parquet"],
                         "test": ["ja/test/*/*.parquet"]}),
    "fleurs": dict(repo="google/fleurs", text="raw_transcription",
                   splits={"train": ["parquet-data/ja_jp/train-*"], "valid": ["parquet-data/ja_jp/validation-*"],
                           "test": ["parquet-data/ja_jp/test-*"]}),
    "jsut": dict(repo="japanese-asr/ja_asr.jsut_basic5000", text="transcription",
                 splits={"test": ["**/*.parquet"]}),
    # ReazonSpeech is distributed with a loading script / tar files -> "stream" mode via `datasets`
    "reazonspeech_small": dict(mode="stream", repo="reazon-research/reazonspeech", config="small",
                               text="transcription", splits={"train": "train"}),
    "reazonspeech_medium": dict(mode="stream", repo="reazon-research/reazonspeech", config="medium",
                                text="transcription", splits={"train": "train"}),
}


# ------------------------------------------------------------------ audio helpers
def to_mono16k_array(x: np.ndarray, sr: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:
        x = x.mean(axis=1)
    return soxr.resample(x, sr, SR, quality="HQ") if sr != SR else x


def decode_audio_bytes(b: bytes) -> np.ndarray:
    x, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=True)
    return to_mono16k_array(x, sr)


def encode_flac(x: np.ndarray) -> bytes:
    buf = io.BytesIO()
    sf.write(buf, np.clip(x, -1.0, 1.0), SR, format="FLAC", subtype="PCM_16")
    return buf.getvalue()


def decode_flac(b: bytes) -> np.ndarray:
    x, _ = sf.read(io.BytesIO(b), dtype="int16")
    return x


class ShardWriter:
    """Writes (audio, text) rows into a parquet shard with ROW_GROUP-sized row groups (atomic rename at close)."""

    def __init__(self, path: str):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        self.path, self.tmp = path, path + ".tmp"
        self.writer = pq.ParquetWriter(self.tmp, SCHEMA, compression="none")  # FLAC is already compressed
        self.rows = {k: [] for k in SCHEMA.names}
        self.n, self.hours = 0, 0.0

    def add(self, x: np.ndarray, text: str, source: str):
        self.rows["audio"].append(encode_flac(x))
        self.rows["text"].append(text)
        self.rows["duration"].append(len(x) / SR)
        self.rows["source"].append(source)
        self.n += 1
        self.hours += len(x) / SR / 3600
        if len(self.rows["audio"]) >= ROW_GROUP:
            self.flush()

    def flush(self):
        if self.rows["audio"]:
            self.writer.write_table(pa.table(self.rows, schema=SCHEMA), row_group_size=ROW_GROUP)
            for v in self.rows.values():
                v.clear()

    def close(self):
        self.flush()
        self.writer.close()
        os.replace(self.tmp, self.path)


# ------------------------------------------------------------------ public datasets
def prepare_parquet_source(name, cfg, splits, raw_dir, out_dir, normalize, min_dur=0.5, max_dur=30.0,
                           max_files=0, download=True):
    """Download parquet files of an HF dataset and convert them to prepared shards. Resumable."""
    from huggingface_hub import snapshot_download

    stats = []
    patterns = [p for s in splits for p in cfg["splits"].get(s, [])]
    if download and patterns:
        print(f"[download] {name}: {cfg['repo']} {patterns}", flush=True)
        snapshot_download(repo_id=cfg["repo"], repo_type="dataset", allow_patterns=patterns,
                          local_dir=os.path.join(raw_dir, name))
    root = os.path.join(raw_dir, name)
    for split in splits:
        files = sorted({f for pat in cfg["splits"].get(split, [])
                        for f in glob.glob(os.path.join(root, "**", "*.parquet"), recursive=True)
                        if fnmatch.fnmatch(os.path.relpath(f, root), pat)})
        if max_files:
            files = files[:max_files]
        for i, f in enumerate(files):
            out = os.path.join(out_dir, split, name, f"{i:05d}.parquet")
            if os.path.exists(out + ".stats.json"):
                stats.append(json.load(open(out + ".stats.json")))
                continue
            st, w = Counter(), ShardWriter(out)
            pf = pq.ParquetFile(f)
            for batch in pf.iter_batches(batch_size=64, columns=["audio", cfg["text"]]):
                for audio, text in zip(batch.column("audio").to_pylist(), batch.column(cfg["text"]).to_pylist()):
                    st["in"] += 1
                    norm = normalize(text or "")
                    b = audio.get("bytes") if isinstance(audio, dict) else None
                    if norm is None or not b:
                        st["skip_text" if norm is None else "skip_no_audio"] += 1
                        continue
                    try:
                        x = decode_audio_bytes(b)
                    except Exception:
                        st["skip_decode"] += 1
                        continue
                    if not (min_dur <= len(x) / SR <= max_dur):
                        st["skip_duration"] += 1
                        continue
                    w.add(x, norm, name)
            w.close()
            rec = {"source": name, "split": split, "file": out, "hours": w.hours, "out": w.n, **st}
            json.dump(rec, open(out + ".stats.json", "w"))
            stats.append(rec)
            print(f"  {name}/{split} {os.path.basename(f)}: kept {w.n}/{st['in']}  ({w.hours:.2f} h)", flush=True)
    return stats


def _example_audio(ex_audio):
    """Handle the audio formats of different `datasets` versions -> (array, sr)."""
    if isinstance(ex_audio, dict):
        if ex_audio.get("array") is not None:
            return np.asarray(ex_audio["array"], dtype=np.float32), ex_audio["sampling_rate"]
        if ex_audio.get("bytes"):
            x, sr = sf.read(io.BytesIO(ex_audio["bytes"]), dtype="float32", always_2d=True)
            return x, sr
    if hasattr(ex_audio, "get_all_samples"):  # datasets>=4 AudioDecoder (torchcodec)
        s = ex_audio.get_all_samples()
        return s.data.numpy().T, s.sample_rate
    raise ValueError("unsupported audio format")


def prepare_stream_source(name, cfg, splits, out_dir, normalize, min_dur=0.5, max_dur=30.0, max_utts=0,
                          shard_utts=20000):
    """Stream a `datasets` dataset (e.g. ReazonSpeech) into prepared shards. Not resumable mid-split."""
    from datasets import load_dataset

    stats = []
    for split in splits:
        hf_split = cfg["splits"].get(split)
        if not hf_split:
            continue
        ds = load_dataset(cfg["repo"], cfg.get("config"), split=hf_split, streaming=True, trust_remote_code=True)
        shard_i, n_total, st, w = 0, 0, Counter(), None
        for ex in ds:
            if max_utts and n_total >= max_utts:
                break
            st["in"] += 1
            norm = normalize(ex.get(cfg["text"]) or "")
            if norm is None:
                st["skip_text"] += 1
                continue
            try:
                x = to_mono16k_array(*_example_audio(ex["audio"]))
            except Exception:
                st["skip_decode"] += 1
                continue
            if not (min_dur <= len(x) / SR <= max_dur):
                st["skip_duration"] += 1
                continue
            if w is None:
                w = ShardWriter(os.path.join(out_dir, split, name, f"{shard_i:05d}.parquet"))
            w.add(x, norm, name)
            n_total += 1
            if w.n >= shard_utts:
                w.close(); stats.append({"source": name, "split": split, "file": w.path, "hours": w.hours, "out": w.n})
                w, shard_i = None, shard_i + 1
        if w is not None:
            w.close(); stats.append({"source": name, "split": split, "file": w.path, "hours": w.hours, "out": w.n})
        print(f"  {name}/{split}: kept {n_total}/{st['in']}  {dict(st)}", flush=True)
    return stats


# ------------------------------------------------------------------ local customer-support corpus
AUDIO_EXT = (".wav", ".flac", ".mp3", ".ogg", ".m4a")


def read_local_corpus(path: str) -> list[dict]:
    """A manifest (.csv/.tsv with path,text[,call_id,speaker,split]) or a folder of audio + .txt pairs."""
    rows = []
    if os.path.isfile(path):
        base = os.path.dirname(path)
        delim = "\t" if path.endswith(".tsv") else ","
        with open(path, encoding="utf-8") as f:
            for r in csv.DictReader(f, delimiter=delim):
                p = r["path"] if os.path.isabs(r["path"]) else os.path.join(base, r["path"])
                rows.append({"path": p, "text": r["text"], "call_id": r.get("call_id") or p,
                             "speaker": r.get("speaker", ""), "split": (r.get("split") or "").strip()})
    else:
        for p in sorted(glob.glob(os.path.join(path, "**", "*"), recursive=True)):
            if p.lower().endswith(AUDIO_EXT) and os.path.exists(os.path.splitext(p)[0] + ".txt"):
                text = open(os.path.splitext(p)[0] + ".txt", encoding="utf-8").read().strip()
                rows.append({"path": p, "text": text, "call_id": os.path.dirname(p), "speaker": "", "split": ""})
    return rows


def split_by_group(group: str, valid_frac: float = 0.05, test_frac: float = 0.05) -> str:
    """Deterministic train/valid/test assignment by call id (no call is split across sets)."""
    u = (zlib.crc32(group.encode()) % 10000) / 10000
    return "test" if u < test_frac else "valid" if u < test_frac + valid_frac else "train"


def prepare_local_corpus(path, out_dir, normalize, source="support", min_dur=0.3, max_dur=30.0,
                         valid_frac=0.05, test_frac=0.05):
    rows = read_local_corpus(path)
    writers, st = {}, Counter()
    for r in rows:
        norm = normalize(r["text"])
        if norm is None:
            st["skip_text"] += 1
            continue
        try:
            x, sr = sf.read(r["path"], dtype="float32", always_2d=True)
            x = to_mono16k_array(x, sr)  # 8 kHz telephone audio is upsampled to 16 kHz here
        except Exception:
            st["skip_decode"] += 1
            continue
        if not (min_dur <= len(x) / SR <= max_dur):
            st["skip_duration"] += 1
            continue
        split = r["split"] or split_by_group(r["call_id"], valid_frac, test_frac)
        if split not in writers:
            writers[split] = ShardWriter(os.path.join(out_dir, split, source, "00000.parquet"))
        writers[split].add(x, norm, source)
    stats = []
    for split, w in writers.items():
        w.close()
        stats.append({"source": source, "split": split, "file": w.path, "hours": w.hours, "out": w.n})
    print(f"[local] {source}: {len(rows)} rows -> {sum(s['out'] for s in stats)} kept {dict(st)}")
    return stats


# ------------------------------------------------------------------ synthetic audio (DRY_RUN only)
def synth_tone_audio(text: str, char_s: float = 0.12, gap_s: float = 0.02, seed: int = 0) -> np.ndarray:
    """Deterministic 'speech' for pipeline tests: each character becomes a short two-tone chirp."""
    rng = np.random.default_rng(seed)
    t = np.arange(int(char_s * SR)) / SR
    env = np.hanning(len(t)).astype(np.float32)
    parts = [np.zeros(int(0.15 * SR), np.float32)]
    for c in text:
        o = ord(c)
        f1, f2 = 150 + (o % 53) * 60, 300 + ((o // 53) % 31) * 110
        parts.append((0.25 * env * (np.sin(2 * np.pi * f1 * t) + 0.6 * np.sin(2 * np.pi * f2 * t))).astype(np.float32))
        parts.append(np.zeros(int(gap_s * SR), np.float32))
    parts.append(np.zeros(int(0.15 * SR), np.float32))
    x = np.concatenate(parts)
    return (x + rng.normal(0, 0.003, len(x))).astype(np.float32)


# ------------------------------------------------------------------ telephony augmentation
def telephony_augment(x: np.ndarray, rng: np.random.Generator, p_narrowband=0.8, p_mulaw=0.5,
                      snr_db=(10.0, 35.0), gain_db=(-8.0, 3.0)) -> np.ndarray:
    """Make wideband 16 kHz audio sound like a phone call: 8 kHz band limit, G.711 mu-law, noise, gain."""
    y = x.astype(np.float32)
    n = len(y)
    if rng.random() < p_narrowband:
        y = soxr.resample(soxr.resample(y, SR, 8000), 8000, SR)
        y = np.pad(y, (0, max(0, n - len(y))))[:n]
    if rng.random() < p_mulaw:
        mu = 255.0
        c = np.sign(y) * np.log1p(mu * np.abs(np.clip(y, -1, 1))) / np.log1p(mu)
        c = np.round((c + 1) / 2 * mu) / mu * 2 - 1  # 8-bit quantization
        y = np.sign(c) * ((1 + mu) ** np.abs(c) - 1) / mu
    if snr_db:
        power = float(np.mean(y**2)) + 1e-10
        snr = rng.uniform(*snr_db)
        y = y + rng.normal(0.0, np.sqrt(power / 10 ** (snr / 10)), n)
    y = y * 10 ** (rng.uniform(*gain_db) / 20)
    return np.clip(y, -1.0, 1.0).astype(np.float32)


# ------------------------------------------------------------------ loaders
def list_shards(data_dir: str, split: str, sources: list[str] | None = None) -> dict[str, list[str]]:
    """Returns {source: [parquet files]} for data_dir/<split>/<source>/*.parquet."""
    out = {}
    for src_dir in sorted(glob.glob(os.path.join(data_dir, split, "*"))):
        name = os.path.basename(src_dir)
        if sources and name not in sources:
            continue
        files = sorted(glob.glob(os.path.join(src_dir, "*.parquet")))
        if files:
            out[name] = files
    return out


def collate(wavs: list[np.ndarray], targets: list[list[int]] | None, texts: list[str]) -> dict:
    lens = torch.tensor([len(w) for w in wavs], dtype=torch.long)
    wav = torch.zeros(len(wavs), int(lens.max()), dtype=torch.float32)
    for i, w in enumerate(wavs):
        wav[i, : len(w)] = torch.from_numpy(w.astype(np.float32) / 32768.0)
    batch = {"wav": wav, "wav_lens": lens, "texts": texts}
    if targets is not None:
        batch["targets"] = torch.tensor([t for ts in targets for t in ts], dtype=torch.long)
        batch["target_lens"] = torch.tensor([len(ts) for ts in targets], dtype=torch.long)
    return batch


class TrainStream(IterableDataset):
    """Infinite, sharded, shuffled stream of row groups with duration-bucketed dynamic batches
    (`max_batch_seconds` of padded audio per step) and optional on-the-fly telephony augmentation."""

    def __init__(self, files, units_path, rank=0, world_size=1, seed=0, max_batch_seconds=600.0,
                 max_batch_size=512, buffer_utts=4000, source_weights=None, telephony_aug_prob=0.0):
        self.files, self.units_path = files, units_path
        self.rank, self.world_size, self.seed = rank, world_size, seed
        self.max_batch_seconds, self.max_batch_size, self.buffer_utts = max_batch_seconds, max_batch_size, buffer_utts
        self.telephony_aug_prob = telephony_aug_prob
        self.items = []  # (file, row_group), repeated for upsampled sources
        for f in files:
            src = os.path.basename(os.path.dirname(f))
            reps = (source_weights or {}).get(src, 1.0)
            whole, frac = int(reps), reps - int(reps)
            for rg in range(pq.ParquetFile(f).metadata.num_row_groups):
                self.items += [(f, rg)] * whole
                if frac > 0 and random.Random(zlib.crc32(f"{f}:{rg}:{seed}".encode())).random() < frac:
                    self.items.append((f, rg))

    def _batches_from_buffer(self, buf, rng):
        buf.sort(key=lambda r: r[2])
        batches, cur, cur_max = [], [], 0.0
        for r in buf:
            m = max(cur_max, r[2])
            if cur and (m * (len(cur) + 1) > self.max_batch_seconds or len(cur) >= self.max_batch_size):
                batches.append(cur)
                cur, m = [], r[2]
            cur.append(r)
            cur_max = m
        if cur:
            batches.append(cur)
        rng.shuffle(batches)
        return batches

    def _make_batch(self, rows, units, np_rng):
        targets = units.encode([r[1] for r in rows])
        wavs, keep_t, keep_txt = [], [], []
        for (audio, text, dur), tgt in zip(rows, targets):
            if not tgt or len(tgt) > dur * FRAME_RATE - 2:  # CTC needs >= 1 frame per label
                continue
            w = decode_flac(audio)
            if self.telephony_aug_prob and np_rng.random() < self.telephony_aug_prob:
                w = (telephony_augment(w.astype(np.float32) / 32768.0, np_rng) * 32767).astype(np.int16)
            wavs.append(w); keep_t.append(tgt); keep_txt.append(text)
        return collate(wavs, keep_t, keep_txt) if wavs else None

    def __iter__(self):
        from .units import Units

        units = Units.load(self.units_path)
        wi = get_worker_info()
        n_workers, wid = (wi.num_workers, wi.id) if wi else (1, 0)
        gid, total = self.rank * n_workers + wid, self.world_size * n_workers
        epoch = 0
        while True:
            rng = random.Random(self.seed * 1000003 + epoch * 7919 + gid)
            np_rng = np.random.default_rng(self.seed * 1000003 + epoch * 7919 + gid)
            order = list(range(len(self.items)))
            random.Random(self.seed + epoch).shuffle(order)
            mine = [self.items[i] for i in order[gid::total]]
            buf = []
            for f, rg in mine:
                t = pq.ParquetFile(f).read_row_group(rg, columns=["audio", "text", "duration"])
                buf.extend(zip(t.column("audio").to_pylist(), t.column("text").to_pylist(),
                               t.column("duration").to_pylist()))
                if len(buf) >= self.buffer_utts:
                    rng.shuffle(buf)
                    head, buf = buf[: self.buffer_utts], buf[self.buffer_utts:]
                    for b in self._batches_from_buffer(head, rng):
                        if (batch := self._make_batch(b, units, np_rng)) is not None:
                            yield batch
            for b in self._batches_from_buffer(buf, rng):
                if (batch := self._make_batch(b, units, np_rng)) is not None:
                    yield batch
            epoch += 1


class EvalSet:
    """Fixed, length-sorted list of utterances. `telephony=True` gives a deterministic phone-quality copy."""

    def __init__(self, files, max_utts=None, seed=0, telephony=False):
        rows = []
        for f in files:
            t = pq.read_table(f, columns=["audio", "text", "duration"])
            rows += list(zip(t.column("audio").to_pylist(), t.column("text").to_pylist(),
                             t.column("duration").to_pylist()))
        if max_utts and len(rows) > max_utts:
            random.Random(seed).shuffle(rows)
            rows = rows[:max_utts]
        self.rows = sorted(rows, key=lambda r: r[2])
        self.hours = sum(r[2] for r in self.rows) / 3600
        self.telephony, self.seed = telephony, seed

    def _wav(self, i, audio):
        w = decode_flac(audio)
        if self.telephony:
            rng = np.random.default_rng(self.seed + i)
            w = (telephony_augment(w.astype(np.float32) / 32768.0, rng, p_narrowband=1.0, p_mulaw=1.0,
                                   snr_db=(20, 30), gain_db=(0, 0)) * 32767).astype(np.int16)
        return w

    def batches(self, rank=0, world_size=1, max_batch_seconds=600.0):
        idx = list(range(rank, len(self.rows), world_size))
        cur = []
        for i in idx:
            if cur and self.rows[i][2] * (len(cur) + 1) > max_batch_seconds:
                yield collate([self._wav(j, self.rows[j][0]) for j in cur], None, [self.rows[j][1] for j in cur])
                cur = []
            cur.append(i)
        if cur:
            yield collate([self._wav(j, self.rows[j][0]) for j in cur], None, [self.rows[j][1] for j in cur])

### 5.4 Run the preparation

* **Real run**: downloads and converts `CFG["public_sources"]` (+ your local corpus if `CFG["local_corpus"]` is set).
  Start with `max_files_per_source=1` to check the pipeline, then set it to 0. Finished shards are skipped on re-run.
* **DRY_RUN**: writes synthetic "audio" for the sample support sentences (plus simple variations) so the rest of
  the notebook has something to train on.

In [ ]:
from functools import partial
from ja_asr import data as D

normalize = partial(normalize_japanese, digit_policy=CFG["digit_policy"], remove_fillers=CFG["remove_fillers"])
prep_stats = []

if DRY_RUN:
    # Synthetic corpus: sample sentences + recombined fragments; split by "call" like a real support corpus.
    texts = [normalize(t) for _, t in SUPPORT_SENTENCES] + [normalize(p) for p in KEIGO_PHRASES]
    rng = random.Random(0)
    frags = [t[i:i + rng.randint(6, 14)] for t in texts for i in range(0, len(t), 7)]
    texts += [a + b for a, b in zip(rng.sample(frags, len(frags)), rng.sample(frags, len(frags)))][:150]
    texts = [t for t in texts if t]
    writers = {}
    for i, t in enumerate(texts):
        split = "valid" if i % 10 == 3 else "test" if i % 10 == 7 else "train"
        if split not in writers:
            writers[split] = D.ShardWriter(f"{CFG['prepared_dir']}/{split}/synthetic/00000.parquet")
        writers[split].add(D.synth_tone_audio(t, seed=i), t, "synthetic")
    for split, w in writers.items():
        w.close()
        prep_stats.append({"source": "synthetic", "split": split, "hours": w.hours, "out": w.n})
else:
    for name in CFG["public_sources"]:
        cfg = D.PUBLIC_SOURCES[name]
        try:
            if cfg.get("mode") == "stream":
                prep_stats += D.prepare_stream_source(name, cfg, ["train", "valid", "test"], CFG["prepared_dir"],
                                                      normalize, CFG["min_dur"], CFG["max_dur"],
                                                      max_utts=CFG["max_files_per_source"] * 20000)
            else:
                prep_stats += D.prepare_parquet_source(name, cfg, ["train", "valid", "test"], CFG["raw_dir"],
                                                       CFG["prepared_dir"], normalize, CFG["min_dur"],
                                                       CFG["max_dur"], CFG["max_files_per_source"])
        except Exception as e:  # missing access, moved repo, no network ...
            print(f"[warn] {name} failed: {type(e).__name__}: {e}\n       -> check the dataset page / your HF access")
    if CFG["local_corpus"]:
        prep_stats += D.prepare_local_corpus(CFG["local_corpus"], CFG["prepared_dir"], normalize, source="support")

json.dump(prep_stats, open(f"{CFG['prepared_dir']}/manifest.json", "w"), ensure_ascii=False, indent=1)
print(f"{len(prep_stats)} shard groups written")

### 5.5 Transcription guideline for your own call data (summary)

Consistent labels matter more than volume for CTC. Agree on these rules before annotating:

1. **Verbatim speech, kanji-kana mixed** as in normal writing (`お問い合わせ`, not `おといあわせ`).
2. **Numbers as spoken**: `三千円`, `ゼロサン の …` for phone numbers, or digits everywhere + `digit_policy="kanji"`; never mix.
3. **Product / plan names in their official spelling** (from `GLOSSARY`).
4. **Keigo verbatim** (`でございます`, `いたします`): do not "simplify" polite forms.
5. **Fillers**: either always transcribe (`えー`, `あのー`) or never; set `remove_fillers` accordingly.
6. **Overlaps / unintelligible parts**: drop the segment rather than guessing.
7. **PII**: anonymize *audio and text* before data leaves the secure environment (APPI); replace names/numbers in
   the transcript only if the audio segment is also removed or bleeped.
8. Segment calls into **utterances of 1–20 s** (VAD + speaker channel), keep `call_id` for leakage-free splits.

---
## 6. Data audit

Before training, look at what you actually have: hours per source and split, duration distribution, speaking
rate, and the script mix (kanji / hiragana / katakana / Latin). Surprises here (e.g. a corpus with katakana-only
labels, or mostly 20 s+ clips) explain surprises in CER later.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from collections import Counter

rows = []
type_counts = Counter()
for split in ("train", "valid", "test"):
    for src, files in D.list_shards(CFG["prepared_dir"], split).items():
        t = pd.concat([pq.read_table(f, columns=["text", "duration"]).to_pandas() for f in files])
        chars = t.text.str.replace(" ", "").str.len()
        rows.append(dict(split=split, source=src, utts=len(t), hours=t.duration.sum() / 3600,
                         mean_s=t.duration.mean(), chars_per_s=(chars / t.duration).mean()))
        if split == "train":
            for txt in t.text:
                type_counts.update(char_type(c) for c in txt if c != " ")
audit = pd.DataFrame(rows)
display(audit.round(3))
tot = sum(type_counts.values())
print("train script mix:", {k: f"{v / tot:.1%}" for k, v in type_counts.most_common()})

In [ ]:
import warnings
import matplotlib.pyplot as plt
from matplotlib import font_manager

# Use a Japanese-capable font if one is installed (e.g. apt install fonts-noto-cjk); plots keep ASCII labels anyway.
_cjk = [f.name for f in font_manager.fontManager.ttflist if any(k in f.name for k in ("Noto Sans CJK", "IPAex", "Hiragino", "Yu Gothic"))]
if _cjk:
    plt.rcParams["font.family"] = _cjk[0]
warnings.filterwarnings("ignore", message="Glyph .* missing from font")

durs = np.concatenate([pq.read_table(f, columns=["duration"]).column("duration").to_numpy()
                       for fs in D.list_shards(CFG["prepared_dir"], "train").values() for f in fs])
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(durs, bins=40, color="#3b82f6")
ax[0].set(title="train utterance duration", xlabel="seconds", ylabel="count")
ax[1].bar(list(dict(type_counts.most_common()).keys()), list(dict(type_counts.most_common()).values()), color="#10b981")
ax[1].set(title="train characters by script")
plt.tight_layout(); plt.show()

---
## 7. CTC unit vocabulary from Gemma tokens

The output units are **the Gemma tokenizer tokens that occur in your Japanese transcripts** (ids: 0 = blank,
1 = `<unk>`, 2… = kept Gemma tokens). Benefits:

* Gemma's tokenizer has good coverage of kanji, kana and common words, so units are word-pieces rather than single characters → fewer CTC steps.
* Each unit's output row is **initialised from Gemma's own text embedding** of that token ("the model's text space").

Two Japanese-specific checks:

1. **Token rate vs. frame rate.** The encoder emits 25 frames/s; CTC needs at least one frame per unit (plus blanks
   between repeats). Japanese speech is ~7–9 morae/s, and Gemma tokens usually cover 1–3 characters, so this is fine;
   utterances that violate it are skipped automatically.
2. **Glossary coverage.** Product names that are rare in public data may tokenize into pieces that fall below
   `min_count` → `<unk>`. Fix: build the vocabulary over **general + domain transcripts** and add the glossary as
   extra text (`extra_texts`), *before* stage-1 training, so stage 2 can reuse the same `units.json`.

In [ ]:
%%writefile ja_asr/units.py
"""CTC output units = the subset of Gemma tokenizer tokens that occur in the Japanese training text.

Unit ids: 0 = CTC blank, 1 = <unk> (rare Gemma tokens), 2.. = kept Gemma tokens.
With model_id == "dummy" a character tokenizer (token id = code point) is used for offline DRY_RUNs.
"""

import json
from collections import Counter
from types import SimpleNamespace

from .text import tidy_spaces

BLANK, UNK = 0, 1


class CharTokenizer:
    """Offline stand-in for the Gemma tokenizer: one token per character, token id = Unicode code point."""

    def __call__(self, texts, add_special_tokens=False):
        return SimpleNamespace(input_ids=[[ord(c) for c in t if c != " "] for t in texts])

    def decode(self, ids):
        return "".join(chr(i) for i in ids)

    def convert_ids_to_tokens(self, ids):
        return [chr(i) for i in ids]


def load_tokenizer(model_id: str):
    if model_id == "dummy":
        return CharTokenizer()
    from transformers import AutoTokenizer

    return AutoTokenizer.from_pretrained(model_id)


class Units:
    def __init__(self, model_id: str, gemma_ids: list[int]):
        self.model_id = model_id
        self.tokenizer = load_tokenizer(model_id)
        self.gemma_ids = list(gemma_ids)
        self.gemma_to_unit = {g: i + 2 for i, g in enumerate(self.gemma_ids)}

    def __len__(self) -> int:
        return len(self.gemma_ids) + 2

    def tokenize(self, texts: list[str]) -> list[list[int]]:
        # Leading space so the first piece gets the same "▁"-prefixed token in every position.
        return self.tokenizer([" " + t for t in texts], add_special_tokens=False).input_ids

    def encode(self, texts: list[str]) -> list[list[int]]:
        return [[self.gemma_to_unit.get(g, UNK) for g in ids] for ids in self.tokenize(texts)]

    def decode(self, unit_ids: list[int]) -> str:
        gemma = [self.gemma_ids[u - 2] for u in unit_ids if u >= 2]
        return tidy_spaces(self.tokenizer.decode(gemma))

    def pieces(self, text: str) -> list[str]:
        """Human-readable unit pieces of a text (<unk> marked)."""
        ids = self.tokenize([text])[0]
        toks = self.tokenizer.convert_ids_to_tokens(ids)
        return [t if g in self.gemma_to_unit else f"<unk:{t}>" for g, t in zip(ids, toks)]

    def save(self, path: str, extra: dict | None = None) -> None:
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"model_id": self.model_id, "gemma_ids": self.gemma_ids, **(extra or {})}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path: str) -> "Units":
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        return cls(d["model_id"], d["gemma_ids"])

    @classmethod
    def build(cls, model_id, texts, min_count=2, extra_texts=None, batch_size=10_000):
        """extra_texts (e.g. glossary terms) are counted with weight min_count so they are always kept."""
        tokenizer = load_tokenizer(model_id)
        counts: Counter = Counter()

        def consume(batch, weight=1):
            for ids in tokenizer(batch, add_special_tokens=False).input_ids:
                for g in ids:
                    counts[g] += weight

        batch = []
        for t in texts:
            batch.append(" " + t)
            if len(batch) == batch_size:
                consume(batch); batch = []
        if batch:
            consume(batch)
        data_total = sum(counts.values())
        if extra_texts:
            consume([" " + t for t in extra_texts], weight=min_count)
        kept = [g for g, c in counts.most_common() if c >= min_count]
        covered = sum(c for g, c in counts.items() if c >= min_count)
        stats = {"distinct_tokens": len(counts), "kept_tokens": len(kept), "train_tokens": data_total,
                 "token_coverage": covered / max(sum(counts.values()), 1)}
        return cls(model_id, kept), stats

In [ ]:
from ja_asr.units import Units

def iter_train_texts():
    for files in D.list_shards(CFG["prepared_dir"], "train").values():
        for f in files:
            yield from pq.read_table(f, columns=["text"]).column("text").to_pylist()

glossary_texts = [normalize(t) for t in GLOSSARY] + [normalize(p) for p in KEIGO_PHRASES]
glossary_texts = [t for t in glossary_texts if t]
units, vocab_stats = Units.build(CFG["encoder_id"], iter_train_texts(), min_count=CFG["min_count"],
                                 extra_texts=glossary_texts)
UNITS_PATH = f"{CFG['prepared_dir']}/units.json"
units.save(UNITS_PATH, extra={"stats": vocab_stats})
print(f"wrote {UNITS_PATH}: {len(units)} units (incl. blank + unk)")
for k, v in vocab_stats.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
print("most frequent:", units.tokenizer.convert_ids_to_tokens(units.gemma_ids[:30]))

In [ ]:
# Glossary / keigo coverage and token rate check
cov = pd.DataFrame([{"term": t, "pieces": " | ".join(units.pieces(normalize(t) or t)),
                     "n_units": len(units.pieces(normalize(t) or t)),
                     "unk": sum(p.startswith("<unk") for p in units.pieces(normalize(t) or t))}
                    for t in list(GLOSSARY) + KEIGO_PHRASES[:5]])
display(cov)

tok_rates = []
for files in D.list_shards(CFG["prepared_dir"], "train").values():
    t = pq.read_table(files[0], columns=["text", "duration"]).to_pandas().head(2000)
    for txt, dur, ids in zip(t.text, t.duration, units.encode(t.text.tolist())):
        tok_rates.append(len(ids) / dur)
tok_rates = np.array(tok_rates)
print(f"units/second: median {np.median(tok_rates):.1f}, p99 {np.percentile(tok_rates, 99):.1f} "
      f"(encoder: 25 frames/s; utterances above ~23 units/s are skipped)")

---
## 8. The model: frozen EmbeddingGemma 2 encoder + CTC reader

Identical in structure (and parameter names) to `qnguyen3/gemma-asr`'s `GemmaCTC`, so checkpoints are
interchangeable with the original `scripts/transcribe.py` / `evaluate.py`:

* **`LogMel`** recomputes the Gemma feature extractor on the GPU in batch (the HF extractor is a per-utterance numpy loop).
* **Hooks** collect the subsampler output + all 12 conformer layer outputs (13 hidden states), mixed by a learned softmax.
  (`output_hidden_states` is not used because the last entry is the 1536-d projected output, not the raw layer.)
* **Reader**: LayerNorm → Linear(enc_dim→512) → depthwise conv (positional info) → 6 pre-norm transformer blocks.
* **Head** in Gemma's text-embedding space: `logits = to_text(x) @ [blank, unk, unit_rows]ᵀ + bias`, `unit_rows`
  initialised from (normalised) Gemma token embeddings.
* **SpecAugment** on the log-mels during training.

In DRY_RUN, `load_gemma_parts("dummy")` returns a tiny random conformer stand-in with the same interface.

In [ ]:
%%writefile ja_asr/model.py
"""EmbeddingGemma 2 audio encoder (frozen) + a small CTC "reader" (structure follows qnguyen3/gemma-asr)."""

from contextlib import nullcontext
from dataclasses import asdict, dataclass, field
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------------ features
def _mel_filters(n_fft=512, n_mels=128, sr=16000, fmin=0.0, fmax=8000.0):
    hz2mel = lambda f: 2595 * np.log10(1 + f / 700)  # noqa: E731
    mel2hz = lambda m: 700 * (10 ** (m / 2595) - 1)  # noqa: E731
    freqs = np.linspace(0, sr / 2, n_fft // 2 + 1)
    pts = mel2hz(np.linspace(hz2mel(fmin), hz2mel(fmax), n_mels + 2))
    fb = np.zeros((n_fft // 2 + 1, n_mels), dtype=np.float32)
    for i in range(n_mels):
        lo, c, hi = pts[i: i + 3]
        fb[:, i] = np.maximum(0, np.minimum((freqs - lo) / (c - lo), (hi - freqs) / (hi - c)))
    return fb


class LogMel(nn.Module):
    """Batched GPU log-mel features numerically matching the EmbeddingGemma 2 (Gemma4) audio feature extractor."""

    def __init__(self, model_id: str):
        super().__init__()
        if model_id == "dummy":
            fe = SimpleNamespace(sampling_rate=16000, frame_length=320, hop_length=160, fft_length=512,
                                 mel_floor=1e-3, window=np.hanning(321)[:-1], mel_filters=_mel_filters())
        else:
            from transformers import AutoFeatureExtractor

            fe = AutoFeatureExtractor.from_pretrained(model_id)
            assert fe.preemphasis == 0.0 and fe.dither == 0.0 and fe.input_scale_factor == 1.0
            assert fe.per_bin_mean is None and fe.per_bin_stddev is None
        self.sampling_rate = fe.sampling_rate
        self.frame_length, self.hop_length, self.fft_length = fe.frame_length, fe.hop_length, fe.fft_length
        self.mel_floor = float(fe.mel_floor)
        self.register_buffer("window", torch.tensor(np.asarray(fe.window), dtype=torch.float32), persistent=False)
        self.register_buffer("mel_filters", torch.tensor(np.asarray(fe.mel_filters), dtype=torch.float32),
                             persistent=False)

    @torch.no_grad()
    def forward(self, wav, wav_lens):
        """wav: (B, L) float32 in [-1, 1], zero padded. Returns features (B, T, 128) and mask (B, T)."""
        pad_left = self.frame_length // 2
        wav = F.pad(wav.float(), (pad_left, 0))
        frames = wav.unfold(-1, self.frame_length + 1, self.hop_length)[..., :-1]
        spec = torch.fft.rfft(frames * self.window, n=self.fft_length, dim=-1).abs()
        feats = torch.log(spec @ self.mel_filters + self.mel_floor)
        frame_end = torch.arange(feats.shape[1], device=wav.device) * self.hop_length + self.frame_length
        mask = frame_end[None, :] < (wav_lens[:, None] + pad_left)
        return feats * mask[..., None], mask


# ------------------------------------------------------------------ stand-in encoder (DRY_RUN)
class _Subsample(nn.Module):
    def __init__(self, n_mels, d):
        super().__init__()
        self.c1, self.c2 = nn.Conv1d(n_mels, d, 3, 2, 1), nn.Conv1d(d, d, 3, 2, 1)

    def forward(self, x):
        return F.gelu(self.c2(F.gelu(self.c1(x.transpose(1, 2))))).transpose(1, 2)


class _Layer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.block = nn.TransformerEncoderLayer(d, 4, 2 * d, dropout=0.0, batch_first=True)

    def forward(self, x, pad_mask):
        return self.block(x, src_key_padding_mask=pad_mask)


class DummyAudioEncoder(nn.Module):
    """Random 'conformer' with the attributes GemmaCTC uses (config, subsample_conv_projection, layers)."""

    def __init__(self, n_mels=128, hidden=128, layers=2):
        super().__init__()
        self.config = SimpleNamespace(num_hidden_layers=layers, hidden_size=hidden)
        self.subsample_conv_projection = _Subsample(n_mels, hidden)  # 100 -> 25 frames/s like the real encoder
        self.layers = nn.ModuleList([_Layer(hidden) for _ in range(layers)])

    def forward(self, feats, mask, return_dict=True):
        x = self.subsample_conv_projection(feats)
        m = mask[:, ::2][:, ::2][:, : x.shape[1]]
        for layer in self.layers:
            x = layer(x, ~m)
        return SimpleNamespace(last_hidden_state=x, attention_mask=m)


# ------------------------------------------------------------------ model
@dataclass
class ModelConfig:
    encoder_id: str = "google/embeddinggemma-2"
    vocab_size: int = 0  # number of CTC units incl. blank and unk
    d_model: int = 512
    n_layers: int = 6
    n_heads: int = 8
    ff_mult: int = 4
    conv_kernel: int = 15
    dropout: float = 0.1
    layer_mix: bool = True
    unfreeze_top_k: int = 0
    head_init: str = "text"
    freeze_head: bool = False
    specaug: dict = field(
        default_factory=lambda: {"freq_masks": 2, "freq_width": 27, "time_masks": 10, "time_width_frac": 0.04}
    )


class TransformerBlock(nn.Module):
    def __init__(self, d, n_heads, ff_mult, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.norm1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.norm2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, ff_mult * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_mult * d, d))
        self.drop = nn.Dropout(dropout)
        self.attn_dropout = dropout

    def forward(self, x, attn_mask):
        B, T, D = x.shape
        q, k, v = self.qkv(self.norm1(x)).view(B, T, 3, self.n_heads, D // self.n_heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                           dropout_p=self.attn_dropout if self.training else 0.0)
        x = x + self.drop(self.proj(a.transpose(1, 2).reshape(B, T, D)))
        return x + self.drop(self.ff(self.norm2(x)))


def load_gemma_parts(encoder_id: str, unit_gemma_ids):
    """Returns (audio_tower, unit token embeddings or None). Drops the rest of EmbeddingGemma 2."""
    if encoder_id == "dummy":  # deterministic: frozen encoder weights are not stored in checkpoints
        torch.manual_seed(0)
        encoder = DummyAudioEncoder()
        emb = torch.randn(len(unit_gemma_ids), 512) if unit_gemma_ids is not None else None
        return encoder, emb
    from transformers import AutoModel

    full = AutoModel.from_pretrained(encoder_id, dtype=torch.bfloat16, vision_config=None)
    emb = None
    if unit_gemma_ids is not None:
        emb = full.language_model.embed_tokens.weight[torch.tensor(unit_gemma_ids)].detach().float().clone()
    encoder = full.audio_tower
    del full
    return encoder, emb


class GemmaCTC(nn.Module):
    def __init__(self, cfg: ModelConfig, encoder: nn.Module, unit_emb=None):
        super().__init__()
        self.cfg = cfg
        self.mel = LogMel(cfg.encoder_id)
        self.encoder = encoder
        n_enc_layers = encoder.config.num_hidden_layers
        enc_dim = encoder.config.hidden_size

        for p in self.encoder.parameters():
            p.requires_grad_(False)
        if cfg.unfreeze_top_k > 0:
            self.encoder.float()  # fp32 master weights for the trained part; forward runs under bf16 autocast
            for layer in self.encoder.layers[n_enc_layers - cfg.unfreeze_top_k:]:
                for p in layer.parameters():
                    p.requires_grad_(True)

        # subsampler output + every conformer layer output, collected with forward hooks
        self._acts = []
        grab = lambda mod, inp, out: self._acts.append(out[0] if isinstance(out, tuple) else out)  # noqa: E731
        self.encoder.subsample_conv_projection.register_forward_hook(grab)
        for layer in self.encoder.layers:
            layer.register_forward_hook(grab)

        d = cfg.d_model
        self.layer_logits = nn.Parameter(torch.zeros(n_enc_layers + 1)) if cfg.layer_mix else None
        self.in_norm = nn.LayerNorm(enc_dim)
        self.in_proj = nn.Linear(enc_dim, d)
        self.pos_conv = nn.Conv1d(d, d, cfg.conv_kernel, padding=cfg.conv_kernel // 2, groups=d)
        self.blocks = nn.ModuleList([TransformerBlock(d, cfg.n_heads, cfg.ff_mult, cfg.dropout)
                                     for _ in range(cfg.n_layers)])
        self.out_norm = nn.LayerNorm(d)

        text_dim = 512 if unit_emb is None else unit_emb.shape[1]
        self.to_text = nn.Linear(d, text_dim)
        n_units = cfg.vocab_size - 2
        if cfg.head_init == "text" and unit_emb is not None:
            rows = F.normalize(unit_emb, dim=-1)
        else:
            rows = torch.randn(n_units, text_dim) * text_dim**-0.5
        self.unit_rows = nn.Parameter(rows, requires_grad=not cfg.freeze_head)
        self.special_rows = nn.Parameter(torch.randn(2, text_dim) * text_dim**-0.5)  # blank, unk
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))

    def _encoder_hidden(self, feats, feat_mask):
        ctx = torch.no_grad() if self.cfg.unfreeze_top_k == 0 else nullcontext()
        enc_dtype = next(self.encoder.parameters()).dtype
        self._acts.clear()
        with ctx:
            out = self.encoder(feats.to(enc_dtype), feat_mask, return_dict=True)
        acts, self._acts = self._acts, []
        assert len(acts) == self.encoder.config.num_hidden_layers + 1
        if self.cfg.layer_mix:
            w = torch.softmax(self.layer_logits.float(), dim=0)
            h = sum(w[i] * a.float() for i, a in enumerate(acts))
        else:
            h = acts[-1].float()
        return h, out.attention_mask

    def train(self, mode: bool = True):
        super().train(mode)
        self.encoder.eval()  # pretrained encoder never uses dropout, even when partially fine-tuned
        return self

    def spec_augment(self, feats, mask):
        sa = self.cfg.specaug
        B, T, Fdim = feats.shape
        lens = mask.sum(1)
        fill = (feats * mask[..., None]).sum(1, keepdim=True) / lens.clamp(min=1)[:, None, None]
        drop = torch.zeros(B, T, Fdim, dtype=torch.bool, device=feats.device)
        f_idx = torch.arange(Fdim, device=feats.device)
        t_idx = torch.arange(T, device=feats.device)
        for _ in range(sa["freq_masks"]):
            w = torch.randint(0, sa["freq_width"] + 1, (B, 1), device=feats.device)
            s = (torch.rand(B, 1, device=feats.device) * (Fdim - w)).long()
            drop |= ((f_idx >= s) & (f_idx < s + w))[:, None, :]
        max_w = (lens.float() * sa["time_width_frac"]).long().clamp(min=1)[:, None]
        for _ in range(sa["time_masks"]):
            w = (torch.rand(B, 1, device=feats.device) * (max_w + 1)).long()
            s = (torch.rand(B, 1, device=feats.device) * (lens[:, None] - w).clamp(min=1)).long()
            drop |= ((t_idx >= s) & (t_idx < s + w))[:, :, None]
        return torch.where(drop, fill.expand_as(feats), feats) * mask[..., None]

    def forward(self, wav, wav_lens, specaug: bool = False):
        """Returns CTC logits (B, T, V) in float32 and output lengths (B,)."""
        feats, feat_mask = self.mel(wav, wav_lens)
        if specaug:
            feats = self.spec_augment(feats, feat_mask)
        h, mask = self._encoder_hidden(feats, feat_mask)
        x = self.in_proj(self.in_norm(h))
        x = x + F.gelu(self.pos_conv((x * mask[..., None]).transpose(1, 2)).transpose(1, 2))
        attn_mask = mask[:, None, None, :]
        for blk in self.blocks:
            x = blk(x, attn_mask)
        t = self.to_text(self.out_norm(x))
        rows = torch.cat([self.special_rows, self.unit_rows], dim=0)
        logits = t.float() @ rows.float().t() + self.head_bias.float()
        return logits, mask.sum(1)

    def trainable_state_dict(self) -> dict:
        """Everything except frozen encoder weights (those come from the HF checkpoint)."""
        trainable = {n for n, p in self.named_parameters() if p.requires_grad}
        return {k: v for k, v in self.state_dict().items() if not k.startswith("encoder.") or k in trainable}

    @classmethod
    def from_checkpoint(cls, ckpt: dict) -> "GemmaCTC":
        cfg = ModelConfig(**ckpt["config"])
        encoder, _ = load_gemma_parts(cfg.encoder_id, None)
        model = cls(cfg, encoder)
        missing, unexpected = model.load_state_dict(ckpt["model"], strict=False)
        missing = [k for k in missing if not k.startswith("encoder.")]
        if missing or unexpected:
            raise RuntimeError(f"bad checkpoint: missing={missing} unexpected={unexpected}")
        return model

    def config_dict(self) -> dict:
        return asdict(self.cfg)


@torch.no_grad()
def greedy_decode(logits, lens):
    """Best-path CTC decoding: argmax, collapse repeats, drop blanks (0)."""
    return [[u for u, _ in seq] for seq in greedy_decode_with_frames(logits, lens)]


@torch.no_grad()
def greedy_decode_with_frames(logits, lens):
    """Like greedy_decode but returns [(unit, frame_index), ...] (frame = 40 ms) for timestamps."""
    best = logits.argmax(-1).cpu()
    out = []
    for b in range(best.shape[0]):
        seq = best[b, : int(lens[b])].tolist()
        res, prev = [], 0
        for t, u in enumerate(seq):
            if u != 0 and u != prev:
                res.append((u, t))
            prev = u
        out.append(res)
    return out

In [ ]:
from ja_asr.model import GemmaCTC, ModelConfig, load_gemma_parts, greedy_decode

encoder, unit_emb = load_gemma_parts(CFG["encoder_id"], units.gemma_ids)
model_cfg = ModelConfig(encoder_id=CFG["encoder_id"], vocab_size=len(units), d_model=CFG["d_model"],
                        n_layers=CFG["n_layers"], n_heads=CFG["n_heads"], dropout=CFG["dropout"],
                        unfreeze_top_k=CFG["unfreeze_top_k"])
model = GemmaCTC(model_cfg, encoder, unit_emb).to(DEVICE)

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"encoder layers: {encoder.config.num_hidden_layers}, hidden: {encoder.config.hidden_size}")
print(f"total params {n_total / 1e6:.1f}M | trainable {n_train / 1e6:.2f}M "
      f"(encoder frozen except top {model_cfg.unfreeze_top_k} layers)")

# Shape check on 3 s of noise
with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=HAS_CUDA):
    logits, out_lens = model.eval()(torch.randn(2, 48000, device=DEVICE) * 0.01,
                                    torch.tensor([48000, 32000], device=DEVICE))
print("logits", tuple(logits.shape), "| frames", out_lens.tolist(), "(expect ~75 and ~50 at 25 frames/s)")

---
## 9. Telephony augmentation

Public corpora are mostly **wideband (16 kHz) read or broadcast speech**; contact-center audio is usually
**narrowband 8 kHz, G.711 μ-law coded**, with line noise and level changes. Training on a mix of clean and
"phone-ified" copies (`CFG["telephony_aug_prob"]`) is one of the cheapest robustness wins for call-center ASR.

`telephony_augment` applies, with random parameters: 16 k → 8 k → 16 k resampling (removes everything above 4 kHz),
8-bit μ-law companding, additive noise at 10–35 dB SNR, and ±gain. The plot compares spectrograms.

In [ ]:
first_train_file = next(iter(D.list_shards(CFG["prepared_dir"], "train").values()))[0]
ex_rows = pq.read_table(first_train_file).slice(0, 1).to_pylist()[0]
x_clean = D.decode_flac(ex_rows["audio"]).astype(np.float32) / 32768
x_phone = D.telephony_augment(x_clean, np.random.default_rng(0), p_narrowband=1.0, p_mulaw=1.0, snr_db=(20, 20))
fig, ax = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for a, x, title in ((ax[0], x_clean, "clean 16 kHz"), (ax[1], x_phone, "telephony-augmented")):
    a.specgram(x, NFFT=512, Fs=16000, noverlap=352, cmap="magma")
    a.set(title=title, xlabel="s")
ax[0].set_ylabel("Hz"); plt.tight_layout(); plt.show()

---
## 10. Scoring: CER and customer-support metrics

* **CER** = (S + D + I) / N on space-stripped, normalized characters — the headline number.
* **SER** (sentence error rate): share of utterances with any error.
* **kana-CER** (optional, needs `pykakasi`): CER after converting both sides to hiragana readings. A large gap
  between CER and kana-CER means the model hears correctly but picks a different kanji/kana spelling
  (`お問い合わせ` vs `お問合せ`) — fix with normalization or label consistency, not more audio.
* **Term recall**: of the glossary terms present in the references, how many appear verbatim in the hypotheses.
  Product names are what agents and analytics care about.
* **Number accuracy**: same for numeral sequences (prices, dates, order numbers).

In [ ]:
%%writefile ja_asr/scoring.py
"""CER / SER / kana-CER / term recall for Japanese ASR (pure Python edit distance, no word segmentation)."""

import re
from collections import Counter
from dataclasses import dataclass, fields

from .text import cer_text, normalize_hypothesis


def edit_ops(ref: str, hyp: str):
    """Levenshtein alignment -> list of (op, ref_char, hyp_char), op in {'=', 'S', 'D', 'I'}."""
    n, m = len(ref), len(hyp)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        ri, row, prev = ref[i - 1], dp[i], dp[i - 1]
        for j in range(1, m + 1):
            row[j] = min(prev[j - 1] + (ri != hyp[j - 1]), prev[j] + 1, row[j - 1] + 1)
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(("=" if ref[i - 1] == hyp[j - 1] else "S", ref[i - 1], hyp[j - 1])); i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            ops.append(("D", ref[i - 1], "")); i -= 1
        else:
            ops.append(("I", "", hyp[j - 1])); j -= 1
    return ops[::-1]


@dataclass
class ErrorCounts:
    sub: int = 0
    dele: int = 0
    ins: int = 0
    ref_chars: int = 0
    utts: int = 0
    sent_err: int = 0

    def __iadd__(self, o):
        for f in fields(self):
            setattr(self, f.name, getattr(self, f.name) + getattr(o, f.name))
        return self

    def summary(self) -> dict:
        n = max(self.ref_chars, 1)
        return {"cer": (self.sub + self.dele + self.ins) / n, "sub": self.sub / n, "del": self.dele / n,
                "ins": self.ins / n, "ser": self.sent_err / max(self.utts, 1), "utts": self.utts,
                "ref_chars": self.ref_chars}


def score_batch(refs, hyps, remove_fillers=False, confusions: Counter | None = None) -> ErrorCounts:
    """refs are normalized transcripts; hyps get normalize_hypothesis. Empty refs are skipped."""
    c = ErrorCounts()
    for r, h in zip(refs, hyps):
        r = cer_text(r)
        if not r:
            continue
        h = cer_text(normalize_hypothesis(h, remove_fillers))
        ops = edit_ops(r, h)
        cnt = Counter(o for o, _, _ in ops)
        c.sub += cnt["S"]; c.dele += cnt["D"]; c.ins += cnt["I"]
        c.ref_chars += len(r); c.utts += 1; c.sent_err += int(r != h)
        if confusions is not None:
            for o, a, b in ops:
                if o != "=":
                    confusions[(o, a, b)] += 1
    return c


_kks = None


def to_kana(text: str) -> str:
    """Hiragana reading via pykakasi (pip install pykakasi)."""
    global _kks
    if _kks is None:
        import pykakasi

        _kks = pykakasi.kakasi()
    return "".join(item["hira"] for item in _kks.convert(text))


def kana_cer(refs, hyps) -> float | None:
    try:
        r = [to_kana(cer_text(x)) for x in refs]
        h = [to_kana(cer_text(normalize_hypothesis(x))) for x in hyps]
    except ImportError:
        return None
    return score_batch(r, h).summary()["cer"]


_NUM = re.compile(r"[〇一二三四五六七八九十百千万億兆点]+|\d+")


def term_recall(refs, hyps, terms) -> dict:
    """For each term: occurrences in refs and how many of those utterances contain it in the hyp."""
    out = {}
    for t in terms:
        n_ref = n_hit = 0
        for r, h in zip(refs, hyps):
            if t in cer_text(r):
                n_ref += 1
                n_hit += t in cer_text(normalize_hypothesis(h))
        if n_ref:
            out[t] = {"in_ref": n_ref, "recognized": n_hit, "recall": n_hit / n_ref}
    return out


def number_accuracy(refs, hyps) -> dict:
    n_ref = n_hit = 0
    for r, h in zip(refs, hyps):
        hh = cer_text(normalize_hypothesis(h))
        for m in _NUM.findall(cer_text(r)):
            if len(m) >= 2 or m.isdigit():
                n_ref += 1
                n_hit += m in hh
    return {"numbers_in_ref": n_ref, "recognized": n_hit, "accuracy": n_hit / max(n_ref, 1)}

In [ ]:
from ja_asr.scoring import score_batch, edit_ops, term_recall, number_accuracy, kana_cer

r, h = "お電話ありがとうございます", "おでんわありがとございます"
print(edit_ops(r, h))
print(score_batch([r], [h]).summary())
print("kana-CER:", kana_cer([r], [h]), "(None = pykakasi not installed)")

---
## 11. Training (stage 1: general Japanese)

Single-process training loop (one GPU, or CPU for the DRY_RUN). It mirrors the original `scripts/train.py`:
AdamW (β=0.9/0.98), linear warm-up + cosine decay, bf16 autocast, CTC loss normalized by target length,
gradient clipping, SpecAugment, duration-bucketed batches — with two Japanese/support changes:

* **model selection by average CER** on the valid sets (the original selects by WER, which is meaningless
  without word boundaries),
* optional **telephony augmentation** of training batches.

Checkpoints (`best.pt`, `last.pt`) use the original format, so `scripts/transcribe.py` from the repo can load them.
For multi-GPU training see Part 19.

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from ja_asr.data import TrainStream, EvalSet, list_shards
from ja_asr.model import greedy_decode
from ja_asr.scoring import ErrorCounts


def lr_lambda(step, warmup, max_steps, min_ratio):
    if step < warmup:
        return (step + 1) / warmup
    p = min(1.0, (step - warmup) / max(1, max_steps - warmup))
    return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * p))


@torch.no_grad()
def evaluate(model, units, eval_sets, max_batch_seconds, remove_fillers=False, return_pairs=False):
    model.eval()
    results, pairs = {}, {}
    for name, es in eval_sets.items():
        counts, refs, hyps = ErrorCounts(), [], []
        for batch in es.batches(max_batch_seconds=max_batch_seconds):
            with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=HAS_CUDA):
                logits, out_lens = model(batch["wav"].to(DEVICE), batch["wav_lens"].to(DEVICE))
            hy = [units.decode(u) for u in greedy_decode(logits, out_lens)]
            counts += score_batch(batch["texts"], hy, remove_fillers)
            refs += batch["texts"]; hyps += hy
        results[name] = counts.summary()
        pairs[name] = (refs, hyps)
    model.train()
    if results:
        results["avg_cer"] = float(np.mean([r["cer"] for r in results.values()]))
    return (results, pairs) if return_pairs else results


def train(model, units, units_path, out_dir, *, max_steps, max_batch_seconds, lr, encoder_lr, warmup_steps,
          weight_decay, grad_clip, min_lr_ratio, telephony_aug_prob, num_workers, log_every, eval_every,
          save_every, max_eval_utts, seed, data_dir, train_sources=None, source_weights=None,
          eval_sources=None, resume=None, **_):
    os.makedirs(out_dir, exist_ok=True)
    torch.manual_seed(seed)
    if HAS_CUDA:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    enc_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("encoder.")]
    other = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.")]
    groups = [{"params": [p for p in other if p.ndim >= 2], "weight_decay": weight_decay, "lr": lr},
              {"params": [p for p in other if p.ndim < 2], "weight_decay": 0.0, "lr": lr}]
    if enc_params:
        groups.append({"params": enc_params, "weight_decay": weight_decay, "lr": encoder_lr})
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8, fused=HAS_CUDA)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: lr_lambda(s, warmup_steps, max_steps, min_lr_ratio))

    step, best = 0, float("inf")
    if resume:
        ckpt = torch.load(resume, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model"], strict=False)
        opt.load_state_dict(ckpt["optimizer"]); sched.load_state_dict(ckpt["scheduler"])
        step, best = ckpt["step"], ckpt.get("best", best)
        print(f"resumed from {resume} at step {step}")

    train_files = [f for fs in list_shards(data_dir, "train", train_sources).values() for f in fs]
    assert train_files, f"no training shards under {data_dir}/train"
    stream = TrainStream(train_files, units_path, seed=seed + step, max_batch_seconds=max_batch_seconds,
                         source_weights=source_weights, telephony_aug_prob=telephony_aug_prob)
    loader = DataLoader(stream, batch_size=None, num_workers=num_workers, pin_memory=HAS_CUDA,
                        prefetch_factor=4 if num_workers else None, persistent_workers=num_workers > 0)
    eval_sets = {n: EvalSet(fs, max_eval_utts, seed=seed)
                 for n, fs in list_shards(data_dir, "valid", eval_sources).items()}
    print(f"train shards: {len(train_files)} | eval: "
          + ", ".join(f"{k} ({v.hours:.2f} h)" for k, v in eval_sets.items()))

    def save(name, extra=None):
        ckpt = {"model": model.trainable_state_dict(), "config": model.config_dict(), "units": units_path,
                "unit_gemma_ids": units.gemma_ids, "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                "step": step, "best": best, "args": {"lang": "ja", "metric": "cer"}, **(extra or {})}
        tmp = os.path.join(out_dir, name + ".tmp")
        torch.save(ckpt, tmp)
        os.replace(tmp, os.path.join(out_dir, name))

    history = []
    metrics_f = open(os.path.join(out_dir, "metrics.jsonl"), "a")
    model.train()
    t_last, audio_acc, loss_acc, n_acc, skipped = time.time(), 0.0, 0.0, 0, 0
    data_iter = iter(loader)
    while step < max_steps:
        batch = next(data_iter)
        wav, lens = batch["wav"].to(DEVICE, non_blocking=True), batch["wav_lens"].to(DEVICE)
        targets, tlens = batch["targets"].to(DEVICE), batch["target_lens"].to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=HAS_CUDA):
            logits, out_lens = model(wav, lens, specaug=True)
        log_probs = F.log_softmax(logits.float(), dim=-1).transpose(0, 1)  # (T, B, V)
        loss = F.ctc_loss(log_probs, targets, out_lens, tlens, blank=0, reduction="sum",
                          zero_infinity=True) / tlens.sum().clamp(min=1)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], grad_clip)
        if torch.isfinite(gnorm):
            opt.step()
        else:
            skipped += 1
        sched.step()
        step += 1
        audio_acc += float(lens.sum()) / 16000; loss_acc += loss.item(); n_acc += 1

        if step % log_every == 0:
            dt = time.time() - t_last
            rec = {"step": step, "loss": loss_acc / n_acc, "lr": sched.get_last_lr()[0], "grad_norm": float(gnorm),
                   "rtf_x": audio_acc / dt, "s_per_step": dt / n_acc, "skipped": skipped}
            if model.layer_logits is not None:
                rec["layer_weights"] = [round(x, 3) for x in torch.softmax(model.layer_logits.float(), 0).tolist()]
            print(f"step {step:6d}  loss {rec['loss']:.3f}  lr {rec['lr']:.2e}  gnorm {rec['grad_norm']:.2f}  "
                  f"{rec['s_per_step']:.2f}s/step  {rec['rtf_x']:.0f}x realtime", flush=True)
            history.append(rec); metrics_f.write(json.dumps(rec) + "\n"); metrics_f.flush()
            t_last, audio_acc, loss_acc, n_acc = time.time(), 0.0, 0.0, 0

        if step % eval_every == 0 or step == max_steps:
            res, pairs = evaluate(model, units, eval_sets, max_batch_seconds, return_pairs=True)
            for name, r in res.items():
                if isinstance(r, dict):
                    ref, hyp = pairs[name][0][-1], pairs[name][1][-1]
                    print(f"  [eval {step}] {name:12s} CER {r['cer']:6.2%}  (S {r['sub']:.1%} D {r['del']:.1%} "
                          f"I {r['ins']:.1%})  SER {r['ser']:.1%}\n      ref: {ref}\n      hyp: {hyp}")
            history.append({"step": step, "eval": res}); metrics_f.write(json.dumps({"step": step, "eval": res}) + "\n")
            if res and res["avg_cer"] < best:
                best = res["avg_cer"]
                save("best.pt", {"eval": res})
                print(f"  new best avg CER {best:.2%} -> {out_dir}/best.pt")
        if step % save_every == 0 or step == max_steps:
            save("last.pt")
    metrics_f.close()
    return history

In [ ]:
train_kwargs = {k: CFG[k] for k in ("max_steps", "max_batch_seconds", "lr", "encoder_lr", "warmup_steps",
                                    "weight_decay", "grad_clip", "min_lr_ratio", "telephony_aug_prob",
                                    "num_workers", "log_every", "eval_every", "save_every", "max_eval_utts", "seed")}
t0 = time.time()
history = train(model, units, UNITS_PATH, CFG["run_dir"], data_dir=CFG["prepared_dir"], **train_kwargs)
print(f"training finished in {(time.time() - t0) / 60:.1f} min")

---
## 12. Training curves and encoder layer weights

The learned softmax weights over the 13 encoder hidden states show **which encoder layers carry the phonetic
information** the reader uses. In the Vietnamese reference run the reader concentrated on middle-to-upper layers;
if your weights are spread almost uniformly after many steps, the reader is under-trained or the learning rate is too low.

In [ ]:
logs = [h for h in history if "loss" in h]
evals = [h for h in history if "eval" in h]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
ax[0].plot([h["step"] for h in logs], [h["loss"] for h in logs], color="#3b82f6")
ax[0].set(title="CTC loss (per target unit)", xlabel="step", yscale="log")
for name in [k for k in (evals[-1]["eval"] if evals else {}) if k != "avg_cer"]:
    ax[1].plot([e["step"] for e in evals], [100 * e["eval"][name]["cer"] for e in evals], marker="o", label=name)
ax[1].set(title="valid CER (%)", xlabel="step"); ax[1].legend()
lw = logs[-1].get("layer_weights") if logs else None
if lw:
    ax[2].bar(range(len(lw)), lw, color="#8b5cf6")
    ax[2].set(title="learned encoder layer weights", xlabel="hidden state (0 = subsampler)")
plt.tight_layout(); plt.show()

---
## 13. Evaluation and error analysis

Evaluate the best checkpoint on every **test** set, in two acoustic conditions (as recorded, and telephone-degraded),
then dig into *what* goes wrong: substitution pairs, term recall for product names, number accuracy and CER by
utterance length. Feed recurring misrecognitions of product names back into `GLOSSARY` (Part 15) and recurring
label inconsistencies back into normalization (Part 3).

In [ ]:
from collections import Counter

ckpt = torch.load(f"{CFG['run_dir']}/best.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model"], strict=False)
print(f"loaded best.pt from step {ckpt['step']} (valid avg CER {ckpt['best']:.2%})")

test_sets = {}
for name, fs in list_shards(CFG["prepared_dir"], "test").items():
    test_sets[name] = EvalSet(fs, None)
    test_sets[name + "@phone"] = EvalSet(fs, None, telephony=True)
res, pairs = evaluate(model, units, test_sets, CFG["max_batch_seconds"], return_pairs=True)

table = []
for name, r in res.items():
    if not isinstance(r, dict):
        continue
    refs, hyps = pairs[name]
    table.append({"test set": name, "utts": r["utts"], "CER %": 100 * r["cer"], "sub %": 100 * r["sub"],
                  "del %": 100 * r["del"], "ins %": 100 * r["ins"], "SER %": 100 * r["ser"],
                  "kana-CER %": (lambda k: None if k is None else 100 * k)(kana_cer(refs, hyps)),
                  "number acc %": 100 * number_accuracy(refs, hyps)["accuracy"]})
display(pd.DataFrame(table).round(2))

In [ ]:
name0 = next(iter(pairs))
refs, hyps = pairs[name0]
conf = Counter()
score_batch(refs, hyps, confusions=conf)
print(f"Top error pairs on {name0}  (op, ref, hyp): count")
for (op, a, b), n in conf.most_common(15):
    print(f"  {op}  {a or '∅'} → {b or '∅'} : {n}")

tr = term_recall(refs, hyps, [normalize(t) for t in GLOSSARY if normalize(t)])
if tr:
    display(pd.DataFrame(tr).T.sort_values("recall"))
else:
    print("(no glossary terms in this test set — add an in-domain test set!)")

# CER by utterance length
len_df = pd.DataFrame({"ref": refs, "hyp": hyps})
len_df["chars"] = len_df.ref.str.replace(" ", "").str.len()
len_df["bucket"] = pd.cut(len_df.chars, [0, 10, 20, 40, 80, 1000])
len_df["cer"] = [score_batch([a], [b]).summary()["cer"] for a, b in zip(len_df.ref, len_df.hyp)]
display(len_df.groupby("bucket", observed=True).cer.agg(["count", "mean"]).round(3))
print("\nexamples:")
for a, b in list(zip(refs, hyps))[:5]:
    print(f"  ref: {a}\n  hyp: {b}\n")

---
## 14. Inference on long recordings

`Transcriber` wraps model + units for production use:

* loads any format `soundfile` can read (wav/flac/ogg/mp3), mixes to mono, resamples to 16 kHz;
* splits long audio into ≤ 30 s chunks **at the quietest 50 ms** near each boundary and decodes chunks in batches;
* returns text plus **segment and token timestamps** (from the CTC frame index × 40 ms);
* reports the real-time factor.

In [ ]:
%%writefile ja_asr/infer.py
"""Batch inference: long-audio chunking, energy VAD, timestamps, stereo call transcription."""

import time

import numpy as np
import soundfile as sf
import soxr
import torch

from .data import collate
from .model import GemmaCTC, greedy_decode_with_frames
from .units import Units

SR = 16000
FRAME_S = 0.04  # encoder frame = 40 ms


def load_audio(path, keep_channels=False):
    x, sr = sf.read(path, dtype="float32", always_2d=True)  # (N, C)
    if not keep_channels:
        x = x.mean(1, keepdims=True)
    if sr != SR:
        x = np.stack([soxr.resample(x[:, c], sr, SR) for c in range(x.shape[1])], 1)
    return x if keep_channels else x[:, 0]


def split_quiet(x, chunk_s=30.0, search_s=5.0):
    """Chunks of at most chunk_s, cut at the lowest-energy 50 ms window in the last search_s. Returns (start, chunk)."""
    chunks, start, n, win = [], 0, int(chunk_s * SR), int(0.05 * SR)
    while len(x) - start > n:
        lo, hi = start + n - int(search_s * SR), start + n
        energy = np.convolve(x[lo:hi] ** 2, np.ones(win), mode="valid")
        cut = lo + int(np.argmin(energy)) + win // 2
        chunks.append((start, x[start:cut]))
        start = cut
    chunks.append((start, x[start:]))
    return [(s, c) for s, c in chunks if len(c) > SR // 10]


def energy_vad(x, frame_ms=30, threshold_db=-35.0, min_speech_s=0.3, min_silence_s=0.4, pad_s=0.15, max_seg_s=25.0):
    """Simple energy VAD -> [(start_sample, end_sample)]. Threshold is relative to the loudest frames."""
    hop = int(SR * frame_ms / 1000)
    n = len(x) // hop
    if n == 0:
        return []
    rms = np.sqrt(np.mean(x[: n * hop].reshape(n, hop) ** 2, axis=1) + 1e-12)
    db = 20 * np.log10(rms)
    speech = db > (np.percentile(db, 95) + threshold_db)
    segs, start, silence = [], None, 0
    for i, s in enumerate(speech):
        if s:
            start = i if start is None else start
            silence = 0
        elif start is not None:
            silence += 1
            if silence * frame_ms / 1000 >= min_silence_s:
                segs.append((start, i - silence + 1)); start, silence = None, 0
    if start is not None:
        segs.append((start, n))
    out, pad = [], int(pad_s * SR)
    for a, b in segs:
        a, b = max(0, a * hop - pad), min(len(x), b * hop + pad)
        if (b - a) / SR >= min_speech_s:
            if (b - a) / SR > max_seg_s:  # long monologue: cut at quiet points
                out += [(a + s, a + s + len(c)) for s, c in split_quiet(x[a:b], max_seg_s, 3.0)]
            else:
                out.append((a, b))
    return out


class Transcriber:
    def __init__(self, model: GemmaCTC, units: Units, device=None, batch_seconds=600.0):
        self.device = device or next(model.parameters()).device
        self.model, self.units, self.batch_seconds = model.eval(), units, batch_seconds

    @classmethod
    def from_checkpoint(cls, path, device=None, **kw):
        device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        units = Units(ckpt["config"]["encoder_id"], ckpt["unit_gemma_ids"])
        model = GemmaCTC.from_checkpoint(ckpt).to(device)
        if device.type == "cpu":
            model.float()
        return cls(model, units, device, **kw)

    @torch.no_grad()
    def transcribe_arrays(self, arrays):
        """List of float32 16 kHz arrays -> [{'text', 'tokens': [(piece, t_seconds), ...]}]."""
        order = sorted(range(len(arrays)), key=lambda i: len(arrays[i]))
        results = [None] * len(arrays)
        cur = []

        def run(idx):
            b = collate([(np.clip(arrays[i], -1, 1) * 32767).astype(np.int16) for i in idx], None, [])
            on_gpu = self.device.type == "cuda"
            with torch.autocast(self.device.type, dtype=torch.bfloat16, enabled=on_gpu):
                logits, lens = self.model(b["wav"].to(self.device), b["wav_lens"].to(self.device))
            for i, seq in zip(idx, greedy_decode_with_frames(logits, lens)):
                units = [u for u, _ in seq]
                toks = [(self.units.decode([u]), round(t * FRAME_S, 2)) for u, t in seq if u >= 2]
                results[i] = {"text": self.units.decode(units), "tokens": toks}

        for i in order:
            if cur and len(arrays[i]) / SR * (len(cur) + 1) > self.batch_seconds:
                run(cur); cur = []
            cur.append(i)
        if cur:
            run(cur)
        return results

    def transcribe(self, audio, chunk_s=30.0):
        """File path or mono array -> {'text', 'segments', 'duration', 'rtf_x'}."""
        x = load_audio(audio) if isinstance(audio, str) else np.asarray(audio, np.float32)
        t0 = time.time()
        chunks = split_quiet(x, chunk_s)
        outs = self.transcribe_arrays([c for _, c in chunks])
        segments = [{"start": round(s / SR, 2), "end": round((s + len(c)) / SR, 2), "text": o["text"],
                     "tokens": [(p, round(s / SR + t, 2)) for p, t in o["tokens"]]}
                    for (s, c), o in zip(chunks, outs) if o["text"]]
        if self.device.type == "cuda":
            torch.cuda.synchronize()
        dt = time.time() - t0
        return {"text": "".join(seg["text"] for seg in segments), "segments": segments,
                "duration": round(len(x) / SR, 2), "rtf_x": round(len(x) / SR / max(dt, 1e-6), 1)}

    def transcribe_call(self, audio, agent_channel=0, labels=("オペレーター", "お客様"), **vad_kw):
        """Stereo call (one speaker per channel) or mono -> time-ordered dialogue turns."""
        x = load_audio(audio, keep_channels=True) if isinstance(audio, str) else np.asarray(audio, np.float32)
        if x.ndim == 1:
            x = x[:, None]
        if x.shape[1] >= 2:
            chans = {labels[0]: x[:, agent_channel], labels[1]: x[:, 1 - agent_channel]}
        else:
            chans = {"話者": x[:, 0]}
        jobs = [(spk, a, b, sig[a:b]) for spk, sig in chans.items() for a, b in energy_vad(sig, **vad_kw)]
        outs = self.transcribe_arrays([j[3] for j in jobs])
        turns = [{"speaker": spk, "start": round(a / SR, 2), "end": round(b / SR, 2), "text": o["text"]}
                 for (spk, a, b, _), o in zip(jobs, outs) if o["text"]]
        return sorted(turns, key=lambda t: t["start"])


def format_dialogue(turns) -> str:
    return "\n".join(f"[{int(t['start'] // 60):02d}:{t['start'] % 60:04.1f}] {t['speaker']}: {t['text']}" for t in turns)

In [ ]:
from ja_asr.infer import Transcriber, format_dialogue

asr = Transcriber.from_checkpoint(f"{CFG['run_dir']}/best.pt", device=DEVICE.type)

if DRY_RUN:
    # a 'long recording': several synthetic utterances concatenated (the stand-in model only knows synthetic audio)
    test_texts = [t for fs in list_shards(CFG["prepared_dir"], "train").values()
                  for f in fs for t in pq.read_table(f, columns=["text"]).column("text").to_pylist()][:4]
    long_audio = np.concatenate([D.synth_tone_audio(t, seed=i) for i, t in enumerate(test_texts)])
    out = asr.transcribe(long_audio, chunk_s=6.0)
    print("reference :", "".join(test_texts))
else:
    # point this at your own audio, e.g. "call_001.wav"
    sample = glob.glob("*.wav") + glob.glob("*.mp3")
    out = asr.transcribe(sample[0]) if sample else {"text": "(put a .wav next to the notebook)", "segments": []}
print("hypothesis:", out["text"])
print(f"duration {out.get('duration')} s, speed {out.get('rtf_x')}x real time")
for seg in out["segments"][:3]:
    print(f"  [{seg['start']:6.2f}–{seg['end']:6.2f}] {seg['text']}   first tokens: {seg['tokens'][:4]}")

---
## 15. Post-processing for customer-support output

CTC output is normalized text (no punctuation, numbers in kanji). For agents, CRM notes and analytics you
usually want:

1. **Inverse text normalization (ITN)** of numbers: `三千円` → `3000円`, `〇三一二三四五六七八` → `0312345678`.
   Only numerals followed by a counter/unit (円, 件, 個, 月, 日, 時, 分, 番, 号, ギガ, …) are converted, and
   idioms like `十分` (enough) or `一番` (most) are left alone.
2. **Glossary correction**: known variants → canonical product names, plus optional fuzzy matching of katakana
   names (`ギガプラント` → `ギガプラン`).
3. **PII masking** (phone, card, postal code, long IDs, e-mail) before transcripts are stored or sent to an LLM.
   Names and addresses need an NER model (e.g. GiNZA / spaCy `ja_ginza`); regexes cannot catch them reliably.

In [ ]:
%%writefile ja_asr/postprocess.py
"""Inverse text normalization (numbers), glossary correction and PII masking for Japanese transcripts."""

import difflib
import re

from .text import hira_to_kata

_DIG = {c: i for i, c in enumerate("〇一二三四五六七八九")}
_DIG["零"] = 0
_SMALL = {"十": 10, "百": 100, "千": 1000}
_BIG = {"万": 10**4, "億": 10**8, "兆": 10**12}
_NUMERAL = "〇零一二三四五六七八九十百千万億兆"
COUNTERS = ("円", "件", "個", "名", "人目", "番目", "番", "号", "階", "月", "日", "時", "分", "秒", "年", "歳",
            "才", "本", "枚", "台", "回", "度", "点", "パーセント", "ギガ", "メガ", "キロ", "グラム", "週間", "ヶ月",
            "か月", "カ月", "時間", "桁", "割", "倍")
IDIOMS = {"十分", "一番", "一度", "一時", "一日中", "一回り", "千円札"}  # leave these alone
_NUM_COUNTER = re.compile(rf"([{_NUMERAL}]+(?:点[{_NUMERAL[:11]}]+)?)(?=({'|'.join(COUNTERS)}))")
_DIGIT_SEQ = re.compile(r"[〇零一二三四五六七八九]{4,}")


def kanji_to_int(s: str) -> int:
    """千五百 -> 1500, 三万五千 -> 35000, 二〇二四 -> 2024."""
    total = section = num = 0
    for c in s:
        if c in _DIG:
            num = num * 10 + _DIG[c]
        elif c in _SMALL:
            section += (num or 1) * _SMALL[c]; num = 0
        elif c in _BIG:
            total += ((section + num) or 1) * _BIG[c]; section = num = 0
    return total + section + num


def _num_str(s: str) -> str:
    if "点" in s:
        a, b = s.split("点", 1)
        return f"{kanji_to_int(a)}." + "".join(str(_DIG[c]) for c in b if c in _DIG)
    return str(kanji_to_int(s))


def itn(text: str) -> str:
    """Kanji numerals -> Arabic digits where it is safe (before counters; digit-by-digit IDs)."""
    text = _DIGIT_SEQ.sub(lambda m: "".join(str(_DIG[c]) for c in m.group()), text)

    def repl(m):
        start, end = m.span(1)
        for idiom in IDIOMS:  # skip if the match is part of an idiom
            i = text.find(idiom, max(0, start - len(idiom)), end + len(idiom))
            if i != -1 and i <= start < i + len(idiom):
                return m.group(1)
        return _num_str(m.group(1))

    return _NUM_COUNTER.sub(repl, text)


def apply_glossary(text: str, glossary: dict, fuzzy: float = 0.0) -> tuple[str, list]:
    """Replace known variants by canonical terms; optionally fuzzy-match katakana terms (ratio >= fuzzy)."""
    fixes = []
    # one longest-first pass: correct spellings match as themselves, variants map to their canonical term
    mapping = {v: k for k, vs in glossary.items() for v in vs if v}
    mapping.update({k: k for k in glossary})
    pattern = re.compile("|".join(map(re.escape, sorted(mapping, key=len, reverse=True))))

    def sub(m):
        canon = mapping[m.group()]
        if canon != m.group():
            fixes.append((m.group(), canon))
        return canon

    text = pattern.sub(sub, text)
    if fuzzy:
        for canon in glossary:
            if len(canon) < 4 or canon in text or not re.fullmatch(r"[ァ-ヶー]+", canon):
                continue
            view = hira_to_kata(text)  # 1:1 character mapping keeps positions aligned
            best = (0.0, None)
            for L in (len(canon) - 1, len(canon), len(canon) + 1):
                for i in range(0, max(0, len(view) - L) + 1):
                    r = difflib.SequenceMatcher(None, view[i:i + L], canon).ratio()
                    if r > best[0]:
                        best = (r, (i, i + L))
            if best[1] and best[0] >= fuzzy:
                i, j = best[1]
                fixes.append((text[i:j], canon)); text = text[:i] + canon + text[j:]
    return text, fixes


PII_PATTERNS = [
    ("カード番号", re.compile(r"(?<!\d)(?:\d[ -]?){13,16}(?!\d)")),
    ("メール", re.compile(r"[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}")),
    ("電話番号", re.compile(r"(?<!\d)0\d{1,4}-?\d{1,4}-?\d{3,4}(?!\d)")),
    ("郵便番号", re.compile(r"(?<!\d)\d{3}-\d{4}(?!\d)")),
    ("番号", re.compile(r"(?<!\d)\d{6,}(?!\d)")),
]


def mask_pii(text: str) -> tuple[str, list]:
    found = []
    for label, pat in PII_PATTERNS:
        for m in pat.finditer(text):
            found.append((label, m.group()))
        text = pat.sub(f"[{label}]", text)
    return text, found


def postprocess(text: str, glossary: dict | None = None, mask: bool = True, fuzzy: float = 0.85) -> dict:
    t = itn(text)
    t, fixes = apply_glossary(t, glossary or {}, fuzzy) if glossary else (t, [])
    masked, pii = mask_pii(t) if mask else (t, [])
    return {"raw": text, "display": t, "masked": masked, "glossary_fixes": fixes, "pii": pii}

In [ ]:
from ja_asr.postprocess import itn, postprocess, kanji_to_int

for s in ["ギガプランは月額三千円でご利用いただけます", "お電話番号は〇三一二三四五六七八ですね",
          "二〇二四年五月三日にご契約です", "料金は千五百円二点五ギガです", "十分ご注意ください一番人気のプランです"]:
    print(f"{s}\n  -> {itn(s)}")
assert kanji_to_int("三万五千") == 35000 and kanji_to_int("千五百") == 1500

print()
demo = "ぎがぷらんに変更したいです電話番号は〇九〇一二三四五六七八です"
print(json.dumps(postprocess(demo, GLOSSARY), ensure_ascii=False, indent=1))
print(postprocess("ギガプラントの料金を教えてください", GLOSSARY)["display"])

---
## 16. Stereo call transcription (operator / customer)

Most contact-center recorders store **one speaker per channel** (agent left, customer right — check yours).
Transcribing each channel separately sidesteps overlap and diarization: VAD finds speech regions per channel,
all regions are decoded in one batch, and the turns are merged by start time into a dialogue.

For mono recordings you need a diarization model (e.g. pyannote) to label speakers; `transcribe_call` then
returns turns labelled `話者`.

In [ ]:
if DRY_RUN:
    # Build a synthetic stereo call from training sentences: agent on channel 0, customer on channel 1.
    train_texts = [t for fs in list_shards(CFG["prepared_dir"], "train").values()
                   for f in fs for t in pq.read_table(f, columns=["text"]).column("text").to_pylist()]
    script = [("agent", train_texts[0]), ("customer", train_texts[1]), ("agent", train_texts[2])]
    call = np.zeros((0, 2), np.float32)
    for spk, t in script:
        x = D.synth_tone_audio(t)
        seg = np.zeros((len(x) + 8000, 2), np.float32)
        seg[: len(x), 0 if spk == "agent" else 1] = x
        call = np.concatenate([call, seg])
    turns = asr.transcribe_call(call, agent_channel=0, threshold_db=-30)
    print("script:\n" + "\n".join(f"  {s}: {t}" for s, t in script))
else:
    stereo = glob.glob("calls/*.wav")
    turns = asr.transcribe_call(stereo[0], agent_channel=0) if stereo else []
print("\nASR dialogue:\n" + format_dialogue([{**t, "text": postprocess(t["text"], GLOSSARY)["masked"]} for t in turns]))

---
## 17. Call analytics on top of the transcript

A transcript becomes useful when it is turned into structured signals. `analyze_call` derives, with transparent
rules you can tune:

* **Intents** (解約 / 請求 / 故障 / 配送 / 契約変更 / 返品 …) from keyword patterns in customer turns,
* **Escalation risk** (`責任者`, `上司`, `消費者センター`, `訴え`, …),
* **Keigo compliance** of the agent: share of agent turns with polite forms + use of standard phrases,
* **Product mentions** from the glossary, **PII** occurrences, and **talk ratio**.

For production you would train a classifier (or call an LLM) on these transcripts; the rules are a baseline and a
source of weak labels.

In [ ]:
%%writefile ja_asr/analytics.py
"""Rule-based analytics for Japanese support-call transcripts (baseline / weak labels)."""

import re
from collections import Counter

INTENT_RULES = {
    "解約": r"解約|退会|やめたい|キャンセル",
    "請求・料金": r"請求|料金|支払|引き落と|口座振替|高い",
    "故障・接続": r"つながらない|繋がらない|故障|動かない|使えな|エラー|圏外",
    "配送": r"届かない|配送|発送|到着|届いて",
    "契約変更": r"プラン.{0,4}変更|変更したい|名義変更|機種変更|住所変更",
    "返品・交換": r"返品|交換|返金",
    "アカウント": r"パスワード|暗証番号|ログイン|マイページ|再設定",
}
ESCALATION = r"責任者|上司|上の人|消費者センター|訴え|弁護士|ふざけ|いい加減|何度も|ありえない"
POLITE = r"です|ます|ございます|いたします|申し訳|恐れ入り|かしこまりました|承り"


def analyze_call(turns, glossary=None, keigo_phrases=None, agent="オペレーター", customer="お客様"):
    cust = [t for t in turns if t["speaker"] == customer] or turns
    ag = [t for t in turns if t["speaker"] == agent]
    cust_text = "".join(t["text"] for t in cust)
    all_text = "".join(t["text"] for t in turns)
    intents = Counter({k: len(re.findall(p, cust_text)) for k, p in INTENT_RULES.items()})
    esc = re.findall(ESCALATION, cust_text)
    polite_turns = sum(bool(re.search(POLITE, t["text"])) for t in ag)
    phrases = [p for p in (keigo_phrases or []) if p in "".join(t["text"] for t in ag)]
    dur = Counter()
    for t in turns:
        dur[t["speaker"]] += t.get("end", 0) - t.get("start", 0)
    total = sum(dur.values()) or 1
    return {
        "intents": [k for k, v in intents.most_common() if v],
        "escalation_risk": "high" if len(esc) >= 2 else "medium" if esc else "low",
        "escalation_terms": esc,
        "agent_keigo_rate": round(polite_turns / max(len(ag), 1), 2),
        "agent_standard_phrases": phrases,
        "products_mentioned": [g for g in (glossary or {}) if g in all_text],
        "talk_ratio": {k: round(v / total, 2) for k, v in dur.items()},
        "n_turns": len(turns),
    }

In [ ]:
from ja_asr.analytics import analyze_call

# Analytics on a reference dialogue (in a real run, use `turns` from Part 16)
demo_turns, t = [], 0.0
for spk, text in SUPPORT_SENTENCES[:12] + SUPPORT_SENTENCES[22:24]:
    d = len(text) * 0.12
    demo_turns.append({"speaker": "オペレーター" if spk == "agent" else "お客様", "start": t, "end": t + d,
                       "text": normalize(text)})
    t += d + 0.5
print(format_dialogue(demo_turns[:4]), "\n...")
print(json.dumps(analyze_call(demo_turns, GLOSSARY, [normalize(p) for p in KEIGO_PHRASES]), ensure_ascii=False, indent=1))

---
## 18. Stage 2: domain adaptation on support calls

After the general model is good (Part 13), continue training on in-domain audio. Recipe that works well for
CTC readers on a frozen encoder:

| Setting | Stage 1 (general) | Stage 2 (support) | Why |
|---|---|---|---|
| data | public corpora | support calls **+ a share of general data** | avoids forgetting general Japanese |
| `source_weights` | – | e.g. `{"support": 4.0}` | upsample the small in-domain set |
| `unfreeze_top_k` | 0 | 1–2 (with ≥ 20 h of domain audio) | adapts the top encoder layers to telephone acoustics |
| `lr` / `encoder_lr` | 1e-3 / – | 3e-4 / 2e-5 | gentle continuation |
| `max_steps` | 10–20k | 2–5k | small data, watch valid CER on the support set |
| `telephony_aug_prob` | 0.5 | 0.3 (real phone audio already) | |
| `units.json` | built on general **+ domain + glossary** text | **reuse** | head rows stay aligned |

Select the checkpoint on the **in-domain valid set** (`eval_sources=["support"]`) and also re-check general test
CER to make sure nothing regressed. Add the in-domain test set to Part 13 and track term recall on product names
across iterations.

**Lighter alternative**: if you need *retrieval* (find calls about X) rather than transcripts, fine-tuning
EmbeddingGemma 2 audio embeddings on (audio, intent/caption) pairs with Sentence Transformers/Unsloth is far less
work than ASR.

In [ ]:
domain_sources = [s for s in list_shards(CFG["prepared_dir"], "train") if s in ("support", "synthetic")]
if not domain_sources:
    print("No in-domain shards found: set CFG['local_corpus'] and re-run Part 5.4 (source name 'support').")
else:
    stage2_cfg = ModelConfig(**{**model_cfg.__dict__, "unfreeze_top_k": 0 if DRY_RUN else 2})
    enc2, _ = load_gemma_parts(CFG["encoder_id"], None)
    model2 = GemmaCTC(stage2_cfg, enc2).to(DEVICE)
    init = torch.load(f"{CFG['run_dir']}/best.pt", map_location="cpu", weights_only=False)
    missing, unexpected = model2.load_state_dict(init["model"], strict=False)
    print("initialised from stage 1; non-encoder missing keys:", [k for k in missing if not k.startswith("encoder.")])
    n2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)
    print(f"stage-2 trainable params: {n2 / 1e6:.2f}M (top {stage2_cfg.unfreeze_top_k} encoder layers unfrozen)")

    stage2_kwargs = {**train_kwargs, "lr": 3e-4, "encoder_lr": 2e-5, "warmup_steps": max(10, CFG["warmup_steps"] // 4),
                     "max_steps": CFG["max_steps"] // 4 if DRY_RUN else 3000, "telephony_aug_prob": 0.3}
    history2 = train(model2, units, UNITS_PATH, CFG["domain_run_dir"], data_dir=CFG["prepared_dir"],
                     source_weights={"support": 4.0}, eval_sources=domain_sources, **stage2_kwargs)

---
## 19. Multi-GPU training with the original repository scripts

For hundreds or thousands of hours (e.g. ReazonSpeech), use the battle-tested DDP scripts of
`qnguyen3/gemma-asr` directly on the shards prepared in this notebook. Three small adaptations are needed for
Japanese:

1. replace `gemma_asr/text.py` with the Japanese normalizer written above (it keeps the function names
   `normalize_text`, `normalize_hypothesis`, `strip_tones` that the repo imports),
2. select the best checkpoint by **CER** instead of WER (one-line patch of `scripts/train.py`),
3. build `units.json` from the Japanese shards (or copy the one created in Part 7, which also covers the glossary).

```bash
git clone https://github.com/qnguyen3/gemma-asr.git && cd gemma-asr
pip install -r requirements.txt
cp ../ja_asr/text.py gemma_asr/text.py                     # Japanese normalization (same API)
sed -i 's/r\["wer"\] for r in results.values()/r["cer"] for r in results.values()/' scripts/train.py   # select by CER
cp ../ja_asr_work/prepared/units.json /path/to/prepared/units.json    # or: python scripts/build_vocab.py --data_dir ...

# single GPU
python scripts/train.py --data_dir ../ja_asr_work/prepared --out_dir runs/ja --max_steps 10000 \
    --max_batch_seconds 600 --num_workers 8
# 4 GPUs
torchrun --nproc_per_node 4 scripts/train.py --data_dir ../ja_asr_work/prepared --out_dir runs/ja \
    --max_steps 20000 --max_batch_seconds 800 --num_workers 10
# domain stage: light encoder unfreezing, upsampled support data, resume weights via --resume or a fresh run
torchrun --nproc_per_node 4 scripts/train.py --data_dir ../ja_asr_work/prepared --out_dir runs/ja_support \
    --unfreeze_top_k 2 --encoder_lr 5e-5 --lr 3e-4 --max_steps 3000 --source_weights '{"support": 4}' \
    --eval_split valid
python scripts/evaluate.py --ckpt runs/ja/best.pt --data_dir ../ja_asr_work/prepared --split test   # read the CER column
python scripts/transcribe.py --ckpt runs/ja/best.pt call_001.wav call_002.mp3
```

The repo logs a "WER" column too; for Japanese text without spaces it is effectively a sentence error rate — use CER.
Checkpoints are interchangeable: `Transcriber.from_checkpoint("gemma-asr/runs/ja/best.pt")` works in this notebook.

In [ ]:
# The patch line, verified against the original train.py source string:
orig = 'results["avg_wer"] = sum(r["wer"] for r in results.values()) / len(results)'
patched = re.sub(r'r\["wer"\] for r in results.values\(\)', 'r["cer"] for r in results.values()', orig)
print(patched)

---
## 20. Serving: LitServe API + MCP tool

This repository deploys models as LitServe endpoints exposed to agents via MCP. The cell below writes a LitServe
server for the trained checkpoint and an MCP bridge, so Claude (or any MCP client) can call
`transcribe_call(path)` and get a masked, post-processed transcript with analytics.

```bash
pip install litserve fastmcp httpx
ASR_CKPT=ja_asr_work/runs/ja_support/best.pt python serving/server_asr_ja.py        # port 8003
curl -s -X POST localhost:8003/predict -H "content-type: application/json" \
     -d '{"path": "call_001.wav", "stereo": true}'
python serving/mcp_asr_bridge.py     # MCP over stdio -> add to claude_desktop_config.json
```

In [ ]:
Path("serving").mkdir(exist_ok=True)
Path("serving/glossary.json").write_text(json.dumps(GLOSSARY, ensure_ascii=False, indent=1), encoding="utf-8")

In [ ]:
%%writefile serving/server_asr_ja.py
"""LitServe endpoint for Japanese customer-support ASR (EmbeddingGemma 2 + CTC reader).

POST /predict  {"path": "call.wav"} or {"audio_b64": "<wav/flac bytes, base64>"},
               optional "stereo": true (agent on channel 0), "mask_pii": true
"""

import base64
import io
import json
import os
import sys
import tempfile

import litserve as ls

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from ja_asr.analytics import analyze_call  # noqa: E402
from ja_asr.infer import Transcriber  # noqa: E402
from ja_asr.postprocess import postprocess  # noqa: E402

CKPT = os.environ.get("ASR_CKPT", "ja_asr_work/runs/ja_general/best.pt")
GLOSSARY = json.load(open(os.path.join(os.path.dirname(__file__), "glossary.json"), encoding="utf-8"))


class JapaneseSupportASR(ls.LitAPI):
    def setup(self, device):
        self.asr = Transcriber.from_checkpoint(CKPT, device="cuda" if "cuda" in str(device) else "cpu")

    def decode_request(self, request):
        if "audio_b64" in request:
            f = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
            f.write(base64.b64decode(request["audio_b64"])); f.close()
            path = f.name
        else:
            path = request["path"]
        return {"path": path, "stereo": bool(request.get("stereo")), "mask": request.get("mask_pii", True)}

    def predict(self, req):
        if req["stereo"]:
            turns = self.asr.transcribe_call(req["path"])
            for t in turns:
                pp = postprocess(t["text"], GLOSSARY, mask=req["mask"])
                t["text"] = pp["masked"] if req["mask"] else pp["display"]
            return {"turns": turns, "analytics": analyze_call(turns, GLOSSARY)}
        out = self.asr.transcribe(req["path"])
        pp = postprocess(out["text"], GLOSSARY, mask=req["mask"])
        return {"text": pp["masked"] if req["mask"] else pp["display"], "duration": out["duration"],
                "rtf_x": out["rtf_x"], "glossary_fixes": pp["glossary_fixes"],
                "pii_found": [label for label, _ in pp["pii"]]}

    def encode_response(self, output):
        return output


if __name__ == "__main__":
    ls.LitServer(JapaneseSupportASR(), accelerator="auto").run(port=8003)

In [ ]:
%%writefile serving/mcp_asr_bridge.py
"""MCP tool bridge for the Japanese support ASR LitServe endpoint (same pattern as examples/mcp_bridge.py)."""

import json

import httpx
from mcp.server.fastmcp import FastMCP

ASR_URL = "http://127.0.0.1:8003"
mcp = FastMCP("ja-support-asr")


@mcp.tool()
def transcribe_call(path: str, stereo: bool = True, mask_pii: bool = True) -> str:
    """Transcribe a Japanese customer-support call recording (wav/flac/mp3). Returns turns + analytics as JSON."""
    r = httpx.post(f"{ASR_URL}/predict", json={"path": path, "stereo": stereo, "mask_pii": mask_pii}, timeout=600.0)
    r.raise_for_status()
    return json.dumps(r.json(), ensure_ascii=False)


if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
# Exercise the server logic in-process (no network needed): decode -> predict -> encode
try:
    import litserve  # noqa: F401
    have_ls = True
except ImportError:
    have_ls = False
print("litserve installed:", have_ls)
if have_ls:
    import importlib.util
    os.environ["ASR_CKPT"] = f"{CFG['run_dir']}/best.pt"
    spec = importlib.util.spec_from_file_location("server_asr_ja", "serving/server_asr_ja.py")
    srv = importlib.util.module_from_spec(spec); spec.loader.exec_module(srv)
    api = srv.JapaneseSupportASR(); api.setup(str(DEVICE))
    import soundfile as sf
    demo_wav = f"{CFG['work_dir']}/demo_call.wav"
    sf.write(demo_wav, call if DRY_RUN else np.zeros((16000, 2), np.float32), 16000)
    print(json.dumps(api.encode_response(api.predict(api.decode_request({"path": demo_wav, "stereo": True}))),
                     ensure_ascii=False, indent=1)[:1500])

---
## 21. Troubleshooting, production notes and references

### Troubleshooting

| Symptom | Likely cause → fix |
|---|---|
| `AutoModel` cannot load `google/embeddinggemma-2` | `transformers<5.19` → upgrade; gated model → accept license + `hf auth login` |
| `AttributeError: audio_tower` / `subsample_conv_projection` | model class layout changed in your transformers version → print `full` and adapt `load_gemma_parts` / hook targets |
| Loss stays at ~ln(V) / output is empty | learning rate too low or warm-up too long; check that targets are not all `<unk>` (Part 7) |
| Many utterances skipped | token rate > 23 units/s (very fast speech or wrong durations) or empty labels after normalization |
| CER good on Common Voice, bad on calls | acoustic mismatch → telephony augmentation, in-domain data, `unfreeze_top_k` 1–2 |
| kana-CER ≪ CER | spelling-variant problem (送り仮名, kanji vs kana) → unify labels / normalization |
| Product names wrong | add domain audio, put terms in `extra_texts`, extend `GLOSSARY` variants |
| Numbers wrong / inconsistent | pick one `digit_policy` for all data; transcribe numbers as spoken |
| Data loader is the bottleneck (`x realtime` low, GPU < 90 %) | raise `num_workers`; keep shards on local NVMe |
| OOM | lower `max_batch_seconds`; keep `unfreeze_top_k=0` |

### Production notes

* **Privacy (APPI / 個人情報保護法)**: call audio and transcripts are personal information. Get consent / notice
  for recording and secondary use (model training), keep data in the agreed region, restrict access, mask PII
  before analytics or LLM calls, and define retention periods. Do not publish models trained on customer calls
  without a legal review (training data can be memorized).
* **Latency**: the model is non-autoregressive; a 30 s chunk is one forward pass. For near-real-time agent assist,
  run VAD-segmented utterances (1–10 s) as they finish; true streaming would need a chunked/causal reader.
* **Punctuation & readability**: CTC output has no punctuation. For agent-facing transcripts add a small
  punctuation model or let an LLM post-edit the *masked* transcript.
* **Monitoring**: log CER on a monthly hand-transcribed sample per queue (billing, technical, …), term recall
  for new products, and the share of empty/low-confidence segments.
* **Model license**: check the EmbeddingGemma 2 terms of use for your deployment.

### References

* Original approach and code: [qnguyen3/gemma-asr](https://github.com/qnguyen3/gemma-asr) (Quan Nguyen, EmbeddingGemma2-ASR-vi)
* Model: [`google/embeddinggemma-2`](https://huggingface.co/google/embeddinggemma-2)
* CTC: Graves et al., *Connectionist Temporal Classification* (ICML 2006); SpecAugment: Park et al. (2019)
* Japanese data: Common Voice ja, FLEURS (`google/fleurs`, `ja_jp`), JSUT (Sonobe et al., 2017),
  ReazonSpeech (Yin et al., 2023), CSJ (Corpus of Spontaneous Japanese, licensed)
* Serving: [LitServe](https://github.com/Lightning-AI/LitServe), [MCP](https://modelcontextprotocol.io)

**Suggested path**: verify the pipeline with a 1-file subset of Common Voice → full Common Voice + FLEURS baseline →
add ReazonSpeech subsets → stage 2 on your own support calls → serve, monitor, iterate on glossary and data.